# Job Satisfaction in Switzerland — Data Preparation

**Dataset:** Swiss Household Panel (SHP), 2004–2024

**Objective:** Prepare and clean the SHP data to examine how job satisfaction differs across occupations, age groups, sex and geographical regions, and how it changes over time.

This notebook documents the data preparation process, from loading the original SAS files to creating a combined dataset covering 21 survey years.

The main steps include:
- Exploring the original datasets and selecting relevant variables.
- Identifying missing values and cleaning the data.
- Merging individual and household information.
- Applying consistent preparation steps across all survey years.
- Selecting the employed respondents for analysis.
- Producing preliminary descriptive statistics and visualisations.

The prepared dataset will be used for further analysis of job satisfaction differences and changes around major economic events.



## 0. Data Source and Research Objective

### 0.1 Research Objective

This project examines how job satisfaction differs across occupational and demographic groups in Switzerland between 2004 and 2024.

We also explore how job satisfaction changed around three major economic events:
- **2009:** Global financial crisis
- **2015:** Swiss franc shock
- **2020:** COVID-19 pandemic

The analysis focuses on differences by occupation, age, sex, employment status and geographical region.

### 0.2 Data Source

The project uses data from the **Swiss Household Panel (SHP)**, a longitudinal survey that follows individuals and households in Switzerland over time.

We use 21 annual survey waves covering 2004–2024. The original data is provided in SAS format (`.sas7bdat`).

Two types of datasets are used:
- **Individual datasets:** Respondent characteristics, employment information, occupation and job satisfaction.
- **Household datasets:** Household information, including geographical region of residence.

These datasets are combined using the household identifier for each survey year.

### 0.3 Scope of the Analysis

The analysis focuses on employed respondents with valid occupation and job satisfaction information.

Employment is identified using the SHP working status variable (`WSTAT`), stored as `WORKING_STATUS` in our prepared dataset. Respondents with `WORKING_STATUS = 1` are included in the employed sample.

Job satisfaction is measured on a scale from **0 to 10**, where higher values indicate greater satisfaction.

The project uses descriptive statistics and visualisations to identify differences between groups and changes over time. These findings will guide the later statistical analysis.

Because the SHP follows respondents across multiple years, the same individual may appear in several survey waves. This longitudinal structure must be considered when interpreting the results.

The analysis is observational, so changes around economic events cannot automatically be interpreted as effects caused by those events.



## 1. Import Libraries

### 1.1 Purpose

Before preparing the SHP data, we import the Python libraries needed for data processing, analysis and visualisation.

The libraries used in this notebook are:

| Library | Purpose |
|---|---|
| `pandas` | Load, clean, merge and analyse datasets using DataFrames |
| `numpy` | Handle numerical operations and missing values |
| `pyreadstat` | Read the original SHP datasets stored in SAS format (`.sas7bdat`) |
| `pathlib` | Locate files and manage folder paths |
| `matplotlib` | Create charts to visualise job satisfaction patterns |
| `seaborn` | Create statistical visualisations, including heatmaps |

### 1.2 Importing the Libraries

The following Python cell imports the libraries required for the data preparation and preliminary descriptive analysis.

Some libraries may also be imported later in the notebook when they are first needed.

In [ ]:
# Import pandas to work with datasets and tables
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import Path to locate files and folders
from pathlib import Path

# Import pyreadstat to read SHP SAS data files
import pyreadstat

# Import matplotlib to create charts
import matplotlib.pyplot as plt

# Import seaborn for statistical visualisations
import seaborn as sns

## 2. Load and Explore the Data

### 2.1 Purpose

Before preparing the full dataset for 2004–2024, we first explore the 2024 SHP data to understand its structure and variables.

We use 2024 as a starting point to test the preparation process before applying it to all 21 survey years.

### 2.2 Loading the Individual Dataset

The individual dataset contains information about respondents, including their age, sex, employment status, occupation and job satisfaction.

We first:
1. Locate the original SHP data files.
2. Load the 2024 individual dataset using `pyreadstat`.
3. Examine the dataset's dimensions, column names and data types.
4. Select the variables relevant to our research questions.
5. Examine the values and missing responses before cleaning.

### 2.3 Initial Data Exploration

We use frequency tables and summary statistics to understand how the variables are recorded.

This helps us identify:
- Valid response categories.
- Missing-value codes.
- Unexpected or invalid values.
- Variables that require cleaning.

At this stage, we are only exploring the data. The cleaning process is documented in Section 3.

In [ ]:
# Define the folder containing the original SHP SAS datasets
data_folder = (
    Path.home()
    / "Documents"
    / "swissubase_932_12_0"
    / "Data_SAS"
    / "SHP-Data-W1-W26-SAS"
)

# Define the folder containing the 2024 survey data
folder_2024 = data_folder / "W26_2024"

# Check whether the folder exists
if not folder_2024.is_dir():
    raise FileNotFoundError(
        f"2024 SHP data folder not found: {folder_2024}"
    )

# Confirm the folder exists
print("2024 data folder exists:", folder_2024.exists())

# Display the folder location
print("Folder location:", folder_2024)

In [ ]:
# Define the file containing the 2024 individual data
person_file_2024 = folder_2024 / "shp24_p_user.sas7bdat"

# Load the 2024 individual dataset
person_2024, meta_2024 = pyreadstat.read_sas7bdat(
    str(person_file_2024)
)

# Display the number of rows and columns
print("Dataset shape:", person_2024.shape)

# Display the first five rows
person_2024.head()

In [ ]:
# Display information about the 2024 individual dataset
# This shows column names, data types and non-missing values
person_2024.info()

In [ ]:
# Select the 2024 variables relevant to our research questions
variables_2024 = [
    "IDPERS",       # Individual identifier
    "IDHOUS24",     # Household identifier
    "AGE24",        # Age
    "SEX24",        # Sex
    "OCCUPA24",     # Employment status
    "WSTAT24",      # Working status
    "IS1MAJ24",     # Occupational group (ISCO)
    "P24W228"       # General job satisfaction
]

# Check whether all required variables exist
missing_variables = [
    variable
    for variable in variables_2024
    if variable not in person_2024.columns
]

if missing_variables:
    raise ValueError(
        f"Missing required variables: {missing_variables}"
    )

print("All 8 required individual variables are available.")

In [ ]:
# Create a smaller dataset with only the variables we need
# .copy() keeps the original 2024 dataset unchanged
person_selected_2024 = person_2024[variables_2024].copy()

# Check the number of rows and columns
print("Selected dataset shape:", person_selected_2024.shape)

# Display the first five rows
person_selected_2024.head()

In [ ]:
# Select the variables we want to examine
# We exclude individual and household IDs because they have many unique values
check_variables = [
    "SEX24",
    "OCCUPA24",
    "WSTAT24",
    "IS1MAJ24",
    "P24W228"
]

# Display the frequency of each value
# This helps us identify valid responses and missing-value codes
for variable in check_variables:
    print("\nVariable:", variable)
    print(person_selected_2024[variable].value_counts(dropna=False).sort_index())

In [ ]:
# Examine the age variable before cleaning
age = person_selected_2024["AGE24"]

# Count missing or invalid ages
valid_age = age.between(0, 120) & age.eq(age.round())

print("Total observations:", len(age))
print("Valid ages:", valid_age.sum())
print("Invalid or missing ages:", (~valid_age).sum())

# Display the minimum and maximum valid age
print("Minimum valid age:", age[valid_age].min())
print("Maximum valid age:", age[valid_age].max())

# Display the values that are not valid ages
print("\nInvalid or missing age values:")
print(age[~valid_age].value_counts(dropna=False))


## 3. Data Cleaning

Before cleaning the dataset, we identified valid values and missing-value codes using the SHP documentation and frequency tables.

Cleaning rules:

- **Age (`AGE24`):** Keep valid integer ages between 0 and 120. Treat negative codes as missing.
- **Sex (`SEX24`):** Keep codes 1 (man), 2 (woman) and 3 (other).
- **Employment status (`OCCUPA24`):** Keep valid categories 1–11 initially. This variable is retained as `EMPLOYMENT_STATUS`.
- **Working status (`WSTAT24`):** Keep codes 1, 2 and 3. Treat negative codes and other invalid values as missing. This variable is stored as `WORKING_STATUS` and used to select the employed sample.
- **Occupational group (`IS1MAJ24`):** Keep ISCO major groups 0–9. Treat negative codes as missing.
- **Job satisfaction (`P24W228`):** Keep whole-number scores from 0 to 10. Treat negative codes and other invalid values as missing.
- **Identifiers (`IDPERS`, `IDHOUS24`):** Preserve these for identifying respondents and merging household data.

We will replace invalid responses with missing values (`NaN`) rather than immediately deleting entire observations. This allows us to examine how much data is available before applying the final sample restrictions.

The original SHP files will remain unchanged.


In [ ]:
# Create a separate copy of the selected 2024 dataset
# This allows us to clean the data without changing the original
person_clean_2024 = person_selected_2024.copy()

# Check that the working copy has the same number of rows and columns
print("Original selected dataset:", person_selected_2024.shape)
print("Working copy for cleaning:", person_clean_2024.shape)

# Display the first five rows of the working copy
person_clean_2024.head()

In [ ]:
# Identify valid ages between 0 and 120
# Ages must also be whole numbers
valid_age = (
    person_clean_2024["AGE24"].between(0, 120)
    & person_clean_2024["AGE24"].eq(
        person_clean_2024["AGE24"].round()
    )
)

# Replace invalid ages with missing values (NaN)
person_clean_2024.loc[~valid_age, "AGE24"] = float("nan")

# Check the results after cleaning
print("Total observations:", len(person_clean_2024))
print("Valid ages:", person_clean_2024["AGE24"].notna().sum())
print("Missing ages:", person_clean_2024["AGE24"].isna().sum())

# Check the minimum and maximum remaining ages
print("Minimum age:", person_clean_2024["AGE24"].min())
print("Maximum age:", person_clean_2024["AGE24"].max())

In [ ]:
# Identify the valid sex categories: 1 (man), 2 (woman), 3 (other)
valid_sex = person_clean_2024["SEX24"].isin([1, 2, 3])

# Replace invalid values with missing values (NaN)
person_clean_2024.loc[~valid_sex, "SEX24"] = float("nan")

# Check the number of valid and missing observations
print("Total observations:", len(person_clean_2024))
print("Valid sex observations:", person_clean_2024["SEX24"].notna().sum())
print("Missing sex observations:", person_clean_2024["SEX24"].isna().sum())

# Display the frequency of each valid category
print("\nSex categories after cleaning:")
print(person_clean_2024["SEX24"].value_counts(dropna=False).sort_index())

In [ ]:
# Identify valid employment status codes from 1 to 11
valid_employment = person_clean_2024["OCCUPA24"].isin(
    range(1, 12)
)

# Replace invalid employment status codes with NaN
person_clean_2024.loc[
    ~valid_employment, "OCCUPA24"
] = float("nan")

# Check the results after cleaning
print("Total observations:", len(person_clean_2024))
print("Valid employment status:", person_clean_2024["OCCUPA24"].notna().sum())
print("Missing employment status:", person_clean_2024["OCCUPA24"].isna().sum())

# Display the frequency of each employment status
print("\nEmployment status after cleaning:")
print(person_clean_2024["OCCUPA24"].value_counts(dropna=False).sort_index())

In [ ]:

# Identify valid working status codes (1, 2 and 3)
valid_working_status = person_clean_2024["WSTAT24"].isin([1, 2, 3])

# Replace invalid working status codes with missing values
person_clean_2024.loc[
    ~valid_working_status, "WSTAT24"
] = float("nan")

# Check the results after cleaning
print("Total observations:", len(person_clean_2024))
print("Valid working status:", person_clean_2024["WSTAT24"].notna().sum())
print("Missing working status:", person_clean_2024["WSTAT24"].isna().sum())

# Display the frequency of each working status code
print("\nWorking status after cleaning:")
print(person_clean_2024["WSTAT24"].value_counts(dropna=False).sort_index())


In [ ]:
# Identify valid ISCO occupational groups from 0 to 9
valid_occupation = person_clean_2024["IS1MAJ24"].isin(
    range(0, 10)
)

# Replace invalid occupation codes with missing values (NaN)
person_clean_2024.loc[
    ~valid_occupation, "IS1MAJ24"
] = float("nan")

# Check the results after cleaning
print("Total observations:", len(person_clean_2024))
print("Valid occupational groups:", person_clean_2024["IS1MAJ24"].notna().sum())
print("Missing occupational groups:", person_clean_2024["IS1MAJ24"].isna().sum())

# Display the frequency of each occupational group
print("\nOccupational groups after cleaning:")
print(person_clean_2024["IS1MAJ24"].value_counts(dropna=False).sort_index())

In [ ]:
# Identify valid job satisfaction scores between 0 and 10
# Scores must also be whole numbers
satisfaction = person_clean_2024["P24W228"]

valid_satisfaction = (
    satisfaction.between(0, 10)
    & satisfaction.eq(satisfaction.round())
)

# Replace invalid job satisfaction scores with NaN
person_clean_2024.loc[
    ~valid_satisfaction, "P24W228"
] = float("nan")

# Check the results after cleaning
print("Total observations:", len(person_clean_2024))
print("Valid job satisfaction:", person_clean_2024["P24W228"].notna().sum())
print("Missing job satisfaction:", person_clean_2024["P24W228"].isna().sum())

# Display the frequency of each valid satisfaction score
print("\nJob satisfaction scores after cleaning:")
print(person_clean_2024["P24W228"].value_counts(dropna=False).sort_index())

In [ ]:
# Check the number of rows and columns after cleaning
print("Cleaned dataset shape:", person_clean_2024.shape)

# Count missing values for each variable
print("\nMissing values by variable:")
print(person_clean_2024.isna().sum())

# Calculate the percentage of missing values
print("\nPercentage missing by variable:")
print(
    (person_clean_2024.isna().mean() * 100).round(2)
)

# Display the first five rows of the cleaned dataset
person_clean_2024.head()

In [ ]:
# Define the file containing the 2024 household data
household_file_2024 = folder_2024 / "shp24_h_user.sas7bdat"

# Load the 2024 household dataset
household_2024, household_meta_2024 = pyreadstat.read_sas7bdat(
    str(household_file_2024)
)

# Display the number of rows and columns
print("Household dataset shape:", household_2024.shape)

# Display the first five rows
household_2024.head()

In [ ]:
# Select the household variables needed for our research
household_variables_2024 = [
    "IDHOUS24",  # Household identifier
    "REGION24"   # Geographic region of residence
]

# Check that both variables exist in the household dataset
for variable in household_variables_2024:
    print(variable, ":", variable in household_2024.columns)

# Create a smaller household dataset without changing the original
household_selected_2024 = household_2024[
    household_variables_2024
].copy()

# Display the number of rows and columns
print("\nSelected household dataset shape:", household_selected_2024.shape)

# Display the first five rows
household_selected_2024.head()

In [ ]:
# Examine the region variable before cleaning
region = household_selected_2024["REGION24"]

# Display the frequency of each region code
print("Region codes:")
print(region.value_counts(dropna=False).sort_index())

# Identify valid region codes from 1 to 7
valid_region = region.isin(range(1, 8))

# Check the number of valid and invalid region values
print("\nTotal households:", len(region))
print("Valid region values:", valid_region.sum())
print("Invalid or missing region values:", (~valid_region).sum())

# Check that each household identifier is unique
print("\nDuplicate household IDs:",
      household_selected_2024["IDHOUS24"].duplicated().sum())

print("Missing household IDs:",
      household_selected_2024["IDHOUS24"].isna().sum())

In [ ]:
# Merge the cleaned individual data with the household region data
# Match individuals and households using IDHOUS24
person_household_2024 = pd.merge(
    person_clean_2024,
    household_selected_2024,
    on="IDHOUS24",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Check the number of rows and columns after merging
print("Merged dataset shape:", person_household_2024.shape)

# Check whether every individual matched a household
print("\nMerge results:")
print(person_household_2024["_merge"].value_counts())

# Check whether any region values are missing
print("\nMissing region values:",
      person_household_2024["REGION24"].isna().sum())

# Display the first five rows
person_household_2024.head()

In [ ]:
# Remove the temporary merge indicator
# It is no longer needed because all individuals matched a household
person_household_2024 = person_household_2024.drop(
    columns="_merge"
)

# Check the final number of rows and columns
print("Dataset shape:", person_household_2024.shape)

# Display the column names
print("\nVariables:")
print(person_household_2024.columns.tolist())

# Display the first five rows
person_household_2024.head()

In [ ]:
# Add the survey year to every observation
person_household_2024["YEAR"] = 2024

# Check the dataset dimensions
print("Dataset shape:", person_household_2024.shape)

# Check that all observations belong to 2024
print("\nSurvey years:")
print(person_household_2024["YEAR"].value_counts())

# Display the first five rows
person_household_2024.head()

In [ ]:
# Rename the 2024 variables to consistent names
person_household_2024 = person_household_2024.rename(
    columns={
        "IDHOUS24": "IDHOUS",
        "AGE24": "AGE",
        "SEX24": "SEX",
        "OCCUPA24": "EMPLOYMENT_STATUS",
        "WSTAT24": "WORKING_STATUS" ,
        "IS1MAJ24": "OCCUPATION",
        "P24W228": "JOB_SATISFACTION",
        "REGION24": "REGION"
    }
)

# Check the updated column names
print("Dataset shape:", person_household_2024.shape)

print("\nColumn names:")
print(person_household_2024.columns.tolist())

# Display the first five rows
person_household_2024.head()

In [ ]:
# Check for duplicate individual identifiers in 2024
duplicate_ids = person_household_2024["IDPERS"].duplicated()

# Display the number of observations and unique individuals
print("Total observations:", len(person_household_2024))
print("Unique individuals:", person_household_2024["IDPERS"].nunique())

# Count duplicate individual identifiers
print("Duplicate individual IDs:", duplicate_ids.sum())

# Check whether any individual identifiers are missing
print("Missing individual IDs:", person_household_2024["IDPERS"].isna().sum())

In [ ]:
# Count missing values for each variable
print("Missing values by variable:")
print(person_household_2024.isna().sum())

# Confirm dataset dimensions
print("\nDataset shape:", person_household_2024.shape)

### 3.1 Validation of the Cleaned and Merged 2024 Dataset

After cleaning the individual variables and merging the household information, we checked whether the resulting dataset was consistent.

The main results were:

- The merged dataset contains **14,582 observations and 10 variables**.
- All **14,582 individual observations** were successfully matched to household information.
- The merge preserved the original number of individual observations.
- The dataset contains no duplicate or missing individual identifiers.
- The variables were renamed consistently to simplify comparisons across survey years.

Invalid responses were recorded as missing values (`NaN`) rather than removing entire observations.

These checks confirm that the 2024 dataset was successfully prepared for the next step: selecting respondents with valid occupation and job satisfaction information.

The same preparation approach will later be applied to the other survey years.

In [ ]:
# Identify respondents with a valid occupational group
has_occupation = person_household_2024["OCCUPATION"].notna()

# Identify respondents with a valid job satisfaction score
has_satisfaction = person_household_2024["JOB_SATISFACTION"].notna()

# Identify respondents with both variables available
has_both = has_occupation & has_satisfaction

# Display the results
print("Total observations:", len(person_household_2024))
print("Valid occupational groups:", has_occupation.sum())
print("Valid job satisfaction scores:", has_satisfaction.sum())
print("Both variables available:", has_both.sum())

# Calculate the percentage of observations with both variables
print(
    "Percentage with both variables:",
    round(has_both.mean() * 100, 2),
    "%"
)

In [ ]:
# Select respondents with both occupation and job satisfaction available
available_sample = person_household_2024.loc[has_both].copy()

# Count respondents in each employment status category
print("Employment status among respondents with both variables:")
print(
    available_sample["EMPLOYMENT_STATUS"]
    .value_counts(dropna=False)
    .sort_index()
)

# Check the total
print("\nTotal respondents:", len(available_sample))

In [ ]:
# Define the employment status categories for our provisional sample
employed_codes = [1, 2, 3, 5]

# Select employed respondents with valid occupation and job satisfaction
employed_sample_2024 = available_sample.loc[
    available_sample["EMPLOYMENT_STATUS"].isin(employed_codes)
].copy()

# Check the size of the provisional employed sample
print("Total respondents in original dataset:", len(person_household_2024))
print("Respondents with both variables:", len(available_sample))
print("Provisional employed sample:", len(employed_sample_2024))

# Check the employment status distribution
print("\nEmployment status distribution:")
print(
    employed_sample_2024["EMPLOYMENT_STATUS"]
    .value_counts()
    .sort_index()
)

# Display the first five rows
employed_sample_2024.head()

In [ ]:
# Check the dimensions of the employed sample
print("Employed sample shape:", employed_sample_2024.shape)

# Count missing values for each variable
print("\nMissing values by variable:")
print(employed_sample_2024.isna().sum())

# Calculate the percentage of missing values
print("\nPercentage missing by variable:")
print(
    (employed_sample_2024.isna().mean() * 100).round(2)
)

In [ ]:
# Select job satisfaction scores from the employed sample
job_satisfaction = employed_sample_2024["JOB_SATISFACTION"]

# Calculate summary statistics
print("Job satisfaction summary:")
print(job_satisfaction.describe().round(2))

# Count respondents for each satisfaction score
print("\nJob satisfaction score distribution:")
print(job_satisfaction.value_counts().sort_index())

# Calculate the average and median
print("\nMean job satisfaction:", round(job_satisfaction.mean(), 2))
print("Median job satisfaction:", job_satisfaction.median())

In [ ]:
# Group respondents by their occupational category
occupation_summary = (
    employed_sample_2024
    .groupby("OCCUPATION")["JOB_SATISFACTION"]
    .agg(["count", "mean", "std"])
    .round(2)
)

# Rename the summary columns
occupation_summary = occupation_summary.rename(
    columns={
        "count": "Respondents",
        "mean": "Mean satisfaction",
        "std": "Standard deviation"
    }
)

# Display the results
print("Job satisfaction by occupational group:")
print(occupation_summary)

In [ ]:
# Group respondents by sex
sex_summary = (
    employed_sample_2024
    .groupby("SEX")["JOB_SATISFACTION"]
    .agg(["count", "mean", "std"])
    .round(2)
)

# Rename the columns
sex_summary = sex_summary.rename(
    columns={
        "count": "Respondents",
        "mean": "Mean satisfaction",
        "std": "Standard deviation"
    }
)

# Replace numeric sex codes with their labels
sex_summary.index = sex_summary.index.map({
    1: "Men",
    2: "Women",
    3: "Other"
})

# Display the results
print("Job satisfaction by sex:")
print(sex_summary)

In [ ]:
# Create age groups for the employed sample
employed_sample_2024["AGE_GROUP"] = pd.cut(
    employed_sample_2024["AGE"],
    bins=[0, 24, 34, 44, 54, 64, 120],
    labels=["0–24", "25–34", "35–44", "45–54", "55–64", "65+"],
    include_lowest=True
)

# Calculate job satisfaction statistics by age group
age_summary = (
    employed_sample_2024
    .groupby("AGE_GROUP", observed=True)["JOB_SATISFACTION"]
    .agg(["count", "mean", "std"])
    .round(2)
)

# Rename the columns
age_summary = age_summary.rename(
    columns={
        "count": "Respondents",
        "mean": "Mean satisfaction",
        "std": "Standard deviation"
    }
)

# Display the results
print("Job satisfaction by age group:")
print(age_summary)

In [ ]:
# Group respondents by their region of residence
region_summary = (
    employed_sample_2024
    .groupby("REGION")["JOB_SATISFACTION"]
    .agg(["count", "mean", "std"])
    .round(2)
)

# Rename the summary columns
region_summary = region_summary.rename(
    columns={
        "count": "Respondents",
        "mean": "Mean satisfaction",
        "std": "Standard deviation"
    }
)

# Display the results
print("Job satisfaction by region of residence:")
print(region_summary)

# Check that all respondents are included
print("\nTotal respondents:", region_summary["Respondents"].sum())

In [ ]:
# Create labels for the employment arrangements
employment_labels = {
    1: "Full-time (37+ hours)",
    2: "Part-time (5–36 hours)",
    3: "Part-time (1–4 hours)",
    5: "Family business"
}

# Calculate job satisfaction by employment arrangement
employment_summary = (
    employed_sample_2024
    .groupby("EMPLOYMENT_STATUS")["JOB_SATISFACTION"]
    .agg(["count", "mean", "std"])
    .round(2)
)

# Rename the summary columns
employment_summary = employment_summary.rename(
    columns={
        "count": "Respondents",
        "mean": "Mean satisfaction",
        "std": "Standard deviation"
    }
)

# Replace employment codes with readable labels
employment_summary.index = employment_summary.index.map(employment_labels)

# Display the results
print("Job satisfaction by employment arrangement:")
print(employment_summary)

In [ ]:
# Count respondents by age group and employment status
age_employment_table = pd.crosstab(
    employed_sample_2024["AGE_GROUP"],
    employed_sample_2024["EMPLOYMENT_STATUS"]
)

# Rename employment status codes
age_employment_table = age_employment_table.rename(
    columns={
        1: "Full-time",
        2: "Part-time (5–36h)",
        3: "Part-time (1–4h)",
        5: "Family business"
    }
)

# Display the number of respondents
print("Employment arrangement by age group:")
print(age_employment_table)

# Calculate percentages within each age group
print("\nPercentage within each age group:")
print(
    (age_employment_table.div(
        age_employment_table.sum(axis=1), axis=0
    ) * 100).round(1)
)

In [ ]:
# Count employed respondents with valid occupation and job satisfaction
sample_2024 = person_household_2024[
    (person_household_2024["WORKING_STATUS"] == 1)
    & person_household_2024["OCCUPATION"].notna()
    & person_household_2024["JOB_SATISFACTION"].notna()
]

print("2024 employed sample:", len(sample_2024))

In [ ]:
# Compare preliminary and final employment samples for 2024

print("Preliminary 2024 sample:", len(employed_sample_2024))
print("Final 2024 employed sample:", len(sample_2024))
print(
    "Additional respondents in final sample:",
    len(sample_2024) - len(employed_sample_2024)
)


## 4. Preliminary Findings — 2024

### 4.1 Purpose

Before preparing the full dataset for 2004–2024, we tested the data preparation process using the 2024 SHP data.

This allowed us to check whether the cleaning rules, household merge and sample selection worked correctly. We also explored job satisfaction across occupation, age, sex, region of residence and employment arrangements.

### 4.2 Data Preparation Results

We completed the following steps:

1. Loaded the 2024 individual and household datasets.
2. Selected and cleaned the 10 variables needed for our research.
3. Replaced invalid responses with missing values (`NaN`).
4. Merged the individual and household datasets using the household identifier.
5. Checked for missing individual identifiers and duplicate respondents.
6. Created a preliminary sample using selected `EMPLOYMENT_STATUS` categories.
7. Created the final employed sample using `WORKING_STATUS == 1` and valid occupation and job satisfaction information.

### 4.3 Key Findings

| Measure | Result |
|---|---:|
| Individuals in the merged 2024 dataset | 14,582 |
| Individuals successfully matched to a household | 14,582 |
| Duplicate individual identifiers | 0 |
| Missing individual identifiers | 0 |
| Preliminary sample (`EMPLOYMENT_STATUS` codes 1, 2, 3, 5) | 5,139 |
| Final employed sample (`WORKING_STATUS == 1`) | 6,007 |
| Additional respondents in the final sample | 868 |

All 14,582 individuals were successfully matched to household information. No duplicate or missing individual identifiers were found.

The preliminary sample contains 5,139 respondents selected using `EMPLOYMENT_STATUS` categories 1, 2, 3 and 5. These represent full-time employment, part-time employment and work in a family business.

We used this preliminary sample to explore job satisfaction across occupation, sex, age, region of residence and employment arrangements.

The final employed sample contains 6,007 respondents selected using `WORKING_STATUS == 1`, with valid occupation and job satisfaction information.

According to the SHP documentation, `WORKING_STATUS` is constructed using individual interview information and is more reliable for identifying employed respondents than `EMPLOYMENT_STATUS`, which comes from the household grid.

The final sample includes 868 additional respondents, including people whose main activity is studying, retirement or another situation but who are also classified as employed.

We therefore use `WORKING_STATUS == 1` for the main analysis. The preliminary results are exploratory and should not be treated as results for the final employed sample.

### 4.4 Next Steps

The 2024 preparation process provides a starting point for preparing the remaining survey years.

In Section 5, we apply consistent cleaning and merging procedures to all 21 survey years from 2004 to 2024.

We then combine the annual datasets, check their consistency and create the final employed analysis sample using `WORKING_STATUS == 1` and valid occupation and job satisfaction information.

The descriptive analysis in Section 6 uses this final sample rather than the preliminary sample.


In [ ]:
# Select the final 2024 employed sample
sample_2024 = person_household_2024[
    (person_household_2024["WORKING_STATUS"] == 1)
    & person_household_2024["OCCUPATION"].notna()
    & person_household_2024["JOB_SATISFACTION"].notna()
].copy()

print("2024 employed sample:", len(sample_2024))

In [ ]:
# Compare employment status and working status for 2024
comparison_2024 = person_household_2024.loc[
    person_household_2024["OCCUPATION"].notna()
    & person_household_2024["JOB_SATISFACTION"].notna()
].copy()

# Show how the two variables overlap
print("Employment status vs working status:")
display(
    pd.crosstab(
        comparison_2024["EMPLOYMENT_STATUS"],
        comparison_2024["WORKING_STATUS"],
        dropna=False,
        margins=True
    )
)

# Compare the two sample selection methods
provisional = comparison_2024[
    comparison_2024["EMPLOYMENT_STATUS"].isin([1, 2, 3, 5])
]

final = comparison_2024[
    comparison_2024["WORKING_STATUS"] == 1
]

print("Provisional sample:", len(provisional))
print("Final sample:", len(final))

# Examine employment categories in the final sample
print("\nEmployment status within the final sample:")
display(
    final["EMPLOYMENT_STATUS"]
    .value_counts(dropna=False)
    .sort_index()
)

# Identify respondents included in one sample but not the other
print(
    "\nOnly in final sample:",
    len(set(final["IDPERS"]) - set(provisional["IDPERS"]))
)

print(
    "Only in provisional sample:",
    len(set(provisional["IDPERS"]) - set(final["IDPERS"]))
)

## 5. Preparing the Full Dataset (2004–2024)

After exploring and cleaning the 2024 SHP data, we now apply the same preparation process to all 21 survey years.

For each year, we will:
1. Load the individual and household datasets.
2. Select the variables needed for our research.
3. Check and clean invalid or missing values.
4. Merge individual and household information.
5. Rename variables consistently across years.
6. Add the survey year.

We will then combine the yearly datasets into one dataset for analysis.

The original SHP files will remain unchanged.

In [ ]:
# Create a list of survey years from 2004 to 2024
years = list(range(2004, 2025))

# Check the number of years
print("Number of survey years:", len(years))

# Display the years
print("Survey years:", years)

In [ ]:
# Create an empty list to store file information
file_check = []

# Check the individual and household files for each year
for year in years:

    # Calculate the SHP wave number
    wave = year - 1998

    # Get the last two digits of the year
    yy = str(year)[-2:]

    # Build the folder path for this year
    year_folder = (
        Path.home()
        / "Documents"
        / "swissubase_932_12_0"
        / "Data_SAS"
        / "SHP-Data-W1-W26-SAS"
        / f"W{wave}_{year}"
    )

    # Build the individual and household file paths
    person_file = year_folder / f"shp{yy}_p_user.sas7bdat"
    household_file = year_folder / f"shp{yy}_h_user.sas7bdat"

    # Record whether both files exist
    file_check.append({
        "YEAR": year,
        "PERSON_FILE": person_file.exists(),
        "HOUSEHOLD_FILE": household_file.exists()
    })

# Convert the results into a table
file_check_df = pd.DataFrame(file_check)

# Display the table
print(file_check_df.to_string(index=False))

# Count years where both files exist
print(
    "\nYears with both files:",
    (file_check_df["PERSON_FILE"] & file_check_df["HOUSEHOLD_FILE"]).sum()
)


### 5.1 Variable Selection

We selected the variables needed to answer our research questions while keeping the dataset manageable.

The individual datasets provide respondent information, employment information, occupation and job satisfaction. The household datasets provide the geographical region.

The final dataset contains 10 variables:

| Variable | Description |
|---|---|
| IDPERS | Individual respondent identifier |
| IDHOUS | Household identifier |
| AGE | Respondent's age |
| SEX | Respondent's sex |
| EMPLOYMENT_STATUS | Employment arrangement or status from OCCUPA |
| WORKING_STATUS | Working status from WSTAT, used to identify employed respondents |
| OCCUPATION | Occupational group based on ISCO major groups |
| JOB_SATISFACTION | Self-reported job satisfaction (0–10) |
| REGION | Geographical region from household data |
| YEAR | Survey year (2004–2024) |

We retain both employment variables because they provide different information. `WORKING_STATUS` is used to select employed respondents, while `EMPLOYMENT_STATUS` is retained for further analysis.

The respondent identifier is also retained to identify individuals appearing in multiple survey years.


In [ ]:
# Create an empty list to store variable checks
variable_checks = []

# Check each survey year
for year in years:

    # Calculate the wave number and year suffix
    wave = year - 1998
    yy = str(year)[-2:]

    # Build the folder path
    year_folder = (
        Path.home()
        / "Documents"
        / "swissubase_932_12_0"
        / "Data_SAS"
        / "SHP-Data-W1-W26-SAS"
        / f"W{wave}_{year}"
    )

    # Build the file paths
    person_file = year_folder / f"shp{yy}_p_user.sas7bdat"
    household_file = year_folder / f"shp{yy}_h_user.sas7bdat"

    # Read only the variable names, without loading the full data
    _, person_metadata = pyreadstat.read_sas7bdat(
        str(person_file),
        metadataonly=True
    )

    _, household_metadata = pyreadstat.read_sas7bdat(
        str(household_file),
        metadataonly=True
    )

    # List the variables needed for this year
    person_variables = [
        "IDPERS",
        f"IDHOUS{yy}",
        f"AGE{yy}",
        f"SEX{yy}",
        f"OCCUPA{yy}",
        f"WSTAT{yy}",
        f"IS1MAJ{yy}",
        f"P{yy}W228"
    ]

    household_variables = [
        f"IDHOUS{yy}",
        f"REGION{yy}"
    ]

    # Identify any variables that are missing
    missing_person = [
        var for var in person_variables
        if var not in person_metadata.column_names
    ]

    missing_household = [
        var for var in household_variables
        if var not in household_metadata.column_names
    ]

    # Store the results
    variable_checks.append({
        "YEAR": year,
        "MISSING_PERSON": missing_person,
        "MISSING_HOUSEHOLD": missing_household
    })

# Convert the results into a table
variable_check_df = pd.DataFrame(variable_checks)

# Display the results
print(variable_check_df.to_string(index=False))


### 5.2 Testing the Preparation Process Using 2004

#### Purpose

Before preparing all 21 survey years, we tested the data preparation process using the 2004 SHP dataset.

This helped us check whether the same cleaning and merging procedures used for 2024 could be applied to earlier survey years.

#### Preparation Steps

For the 2004 dataset, we:

1. Located and loaded the individual and household SAS files.
2. Checked the availability of the required variables, including `WSTAT` for working status.
3. Selected the variables needed for our research.
4. Replaced invalid responses with missing values (`NaN`).
5. Merged the individual and household datasets using the household identifier.
6. Standardised the variable names to match the 2024 dataset.
7. Added the survey year (`YEAR = 2004`).
8. Checked the resulting dataset for consistency.

#### Objective

The goal was to confirm that the preparation process worked across different survey years before automating it for the full 2004–2024 period.

The original SHP files remain unchanged.


In [ ]:
# Select the first survey year
year = 2004

# Calculate its wave number and year suffix
wave = year - 1998
yy = str(year)[-2:]

# Build the folder path
year_folder = (
    Path.home()
    / "Documents"
    / "swissubase_932_12_0"
    / "Data_SAS"
    / "SHP-Data-W1-W26-SAS"
    / f"W{wave}_{year}"
)

# Build the individual file path
person_file = year_folder / f"shp{yy}_p_user.sas7bdat"

# Select only the variables needed for our research
person_variables = [
    "IDPERS",
    f"IDHOUS{yy}",
    f"AGE{yy}",
    f"SEX{yy}",
    f"OCCUPA{yy}",
    f"WSTAT{yy}",
    f"IS1MAJ{yy}",
    f"P{yy}W228"
]

# Load the selected variables
person_2004, metadata_2004 = pyreadstat.read_sas7bdat(
    str(person_file),
    usecols=person_variables
)

# Check the dataset dimensions
print("Dataset shape:", person_2004.shape)

# Display the variable names
print("\nVariables:")
print(person_2004.columns.tolist())

# Preview the first five rows
display(person_2004.head())

In [ ]:
# Select the variables we need to inspect
variables_to_check = [
    "AGE04",
    "SEX04",
    "OCCUPA04",
    "WSTAT04",
    "IS1MAJ04",
    "P04W228"
]

# Examine each variable separately
for variable in variables_to_check:

    # Display the variable name
    print(f"\n--- {variable} ---")

    # Count the number of missing values
    print("Missing (NaN):", person_2004[variable].isna().sum())

    # Show the minimum and maximum values
    print("Minimum:", person_2004[variable].min())
    print("Maximum:", person_2004[variable].max())

    # Display the frequency of each value
    print("Value counts:")
    print(
        person_2004[variable]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )

In [ ]:
# Create a copy so the original 2004 data remains unchanged
person_clean_2004 = person_2004.copy()

# Replace invalid ages with NaN
person_clean_2004.loc[
    ~person_clean_2004["AGE04"].between(0, 120),
    "AGE04"
] = float("nan")

# Keep only valid sex codes
person_clean_2004.loc[
    ~person_clean_2004["SEX04"].isin([1, 2, 3]),
    "SEX04"
] = float("nan")

# Keep only valid employment status codes
person_clean_2004.loc[
    ~person_clean_2004["OCCUPA04"].between(1, 11),
    "OCCUPA04"
] = float("nan")

# Keep only valid working status codes
person_clean_2004.loc[
    ~person_clean_2004["WSTAT04"].isin([1, 2, 3]),
    "WSTAT04"
] = float("nan")

# Keep only valid occupation codes
person_clean_2004.loc[
    ~person_clean_2004["IS1MAJ04"].between(0, 9),
    "IS1MAJ04"
] = float("nan")

# Keep only whole-number job satisfaction scores from 0 to 10
valid_satisfaction = (
    person_clean_2004["P04W228"].between(0, 10)
    & person_clean_2004["P04W228"].eq(
        person_clean_2004["P04W228"].round()
    )
)

person_clean_2004.loc[
    ~valid_satisfaction,
    "P04W228"
] = float("nan")

# Display the number of valid and missing values after cleaning
print("2004 cleaning results:\n")

for variable in [
    "AGE04", "SEX04", "OCCUPA04", "WSTAT04", "IS1MAJ04", "P04W228"
]:
    print(
        variable,
        "| Valid:", person_clean_2004[variable].notna().sum(),
        "| Missing:", person_clean_2004[variable].isna().sum()
    )

# Confirm the dataset dimensions
print("\nDataset shape:", person_clean_2004.shape)

In [ ]:
# Build the path to the 2004 household dataset
household_file = year_folder / "shp04_h_user.sas7bdat"

# Select the household ID and region of residence
household_variables = [
    "IDHOUS04",
    "REGION04"
]

# Load only the selected household variables
household_2004, household_metadata_2004 = pyreadstat.read_sas7bdat(
    str(household_file),
    usecols=household_variables
)

# Display the dataset dimensions
print("Household dataset shape:", household_2004.shape)

# Display the variable names
print("\nVariables:")
print(household_2004.columns.tolist())

# Preview the first five households
display(household_2004.head())

# Check the region codes
print("\nRegion value counts:")
print(
    household_2004["REGION04"]
    .value_counts(dropna=False)
    .sort_index()
)

# Check whether household IDs are unique
print("\nDuplicate household IDs:",
      household_2004["IDHOUS04"].duplicated().sum())

# Check for missing household IDs
print("Missing household IDs:",
      household_2004["IDHOUS04"].isna().sum())

In [ ]:
# Create a cleaned copy of the household dataset
household_clean_2004 = household_2004.copy()

# Keep only valid region codes (1 to 7)
household_clean_2004.loc[
    ~household_clean_2004["REGION04"].between(1, 7),
    "REGION04"
] = float("nan")

# Merge individual and household data using the household ID
person_household_2004 = person_clean_2004.merge(
    household_clean_2004,
    on="IDHOUS04",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Check how many respondents matched a household
print("Merge results:")
print(person_household_2004["_merge"].value_counts())

# Check the dataset dimensions
print("\nDataset shape:", person_household_2004.shape)

# Check for missing regions after merging
print(
    "\nMissing regions:",
    person_household_2004["REGION04"].isna().sum()
)

# Preview the merged dataset
display(person_household_2004.head())

In [ ]:
# Create a copy of the merged dataset
standardised_2004 = person_household_2004.copy()

# Remove the temporary merge-check column
standardised_2004 = standardised_2004.drop(columns=["_merge"])

# Rename variables to consistent names
standardised_2004 = standardised_2004.rename(
    columns={
        "IDHOUS04": "IDHOUS",
        "AGE04": "AGE",
        "SEX04": "SEX",
        "OCCUPA04": "EMPLOYMENT_STATUS",
        "WSTAT04": "WORKING_STATUS" ,
        "IS1MAJ04": "OCCUPATION",
        "P04W228": "JOB_SATISFACTION",
        "REGION04": "REGION"
    }
)

# Add the survey year
standardised_2004["YEAR"] = 2004

# Arrange the columns in a consistent order
standardised_2004 = standardised_2004[
    [
        "IDPERS",
        "IDHOUS",
        "YEAR",
        "AGE",
        "SEX",
        "EMPLOYMENT_STATUS",
        "WORKING_STATUS" ,
        "OCCUPATION",
        "JOB_SATISFACTION",
        "REGION"
    ]
]

# Display the dataset dimensions and column names
print("Dataset shape:", standardised_2004.shape)
print("\nColumns:")
print(standardised_2004.columns.tolist())

# Check for duplicate respondent IDs within 2004
print(
    "\nDuplicate respondent IDs:",
    standardised_2004["IDPERS"].duplicated().sum()
)

# Preview the standardised dataset
display(standardised_2004.head())

In [ ]:

# Check respondents with valid occupation and job satisfaction
has_both_2004 = (
    standardised_2004["OCCUPATION"].notna()
    & standardised_2004["JOB_SATISFACTION"].notna()
)

# Create a sample with both variables available
available_sample_2004 = standardised_2004.loc[
    has_both_2004
].copy()

# Select employed respondents using working status
employed_sample_2004 = available_sample_2004.loc[
    available_sample_2004["WORKING_STATUS"] == 1
].copy()

# Display sample sizes
print("Total respondents:", len(standardised_2004))
print("Valid occupation and satisfaction:", len(available_sample_2004))
print("Employed sample:", len(employed_sample_2004))

# Check working status distribution
print("\nWorking status distribution:")
print(
    employed_sample_2004["WORKING_STATUS"]
    .value_counts()
    .sort_index()
)

# Check missing values in the employed sample
print("\nMissing values:")
print(employed_sample_2004.isna().sum())

# Check average job satisfaction
print(
    "\nMean job satisfaction:",
    round(employed_sample_2004["JOB_SATISFACTION"].mean(), 2)
)


In [ ]:
# Identify respondents appearing in both 2004 and 2024
ids_2004 = set(standardised_2004["IDPERS"].dropna())
ids_2024 = set(person_household_2024["IDPERS"].dropna())

# Find IDs shared between the two years
shared_ids = ids_2004.intersection(ids_2024)

# Display the results
print("Unique respondent IDs in 2004:", len(ids_2004))
print("Unique respondent IDs in 2024:", len(ids_2024))
print("Respondent IDs in both years:", len(shared_ids))

# Check missing respondent IDs
print(
    "\nMissing respondent IDs in 2004:",
    standardised_2004["IDPERS"].isna().sum()
)

print(
    "Missing respondent IDs in 2024:",
    person_household_2024["IDPERS"].isna().sum()
)


#### Interpretation — Testing the Preparation Process Using 2004

The 2004 dataset was successfully loaded, cleaned, merged and standardised using the same approach developed for 2024.

The main results were:

- The original individual dataset contained **14,076 respondents**.
- All 14,076 respondents were successfully matched with household information.
- No region values were missing after the merge.
- The standardised dataset contains **10 variables**, consistent with the 2024 dataset.
- No duplicate respondent IDs were found in the 2004 dataset.
- No respondent identifiers were missing in either year.
- A total of **3,799 respondents** appeared in both the 2004 and 2024 datasets.
- The final employed sample for 2004 contains **5,421 observations**, with an average job satisfaction of **8.21 out of 10**.

The shared respondent IDs confirm that some individuals can be followed across survey years, which is important for understanding the longitudinal structure of the SHP.

Overall, the 2004 test confirms that the preparation process can be applied to an earlier survey year. This provides a basis for automating the same steps across all 21 years (2004–2024).


### 5.3 Automating Data Preparation Across Years

#### Purpose

After testing the preparation process using 2004 and 2024, we create a Python function to apply the same steps to other survey years.

This reduces repeated code and helps ensure that the datasets are prepared consistently.

#### Preparation Function

The function `prepare_year(year)` is designed to:

1. Identify the individual and household SAS files for the selected year.
2. Load both datasets.
3. Select the variables required for our research.
4. Apply the cleaning rules established earlier.
5. Merge individual and household information.
6. Standardise variable names across survey years.
7. Add a `YEAR` column to identify the survey year.

#### Why Automation Is Useful

The SHP datasets use year-specific variable names, such as `AGE04` for 2004 and `AGE24` for 2024.

Standardising these names allows us to combine the annual datasets into one dataset with consistent columns.

The function also makes it easier to check and correct the preparation process without manually repeating every step for each year.

The first function, `get_year_files(year)`, checks the expected file paths for a selected year. The complete `prepare_year(year)` function then loads, cleans, merges and standardises the annual dataset.


In [ ]:
# Define a function to prepare data for one survey year
def get_year_files(year):

    # Calculate the SHP wave number
    wave = year - 1998

    # Get the last two digits of the year
    yy = str(year)[-2:]

    # Build the folder path for the selected year
    year_folder = (
        Path.home()
        / "Documents"
        / "swissubase_932_12_0"
        / "Data_SAS"
        / "SHP-Data-W1-W26-SAS"
        / f"W{wave}_{year}"
    )

    # Build the individual and household file paths
    person_file = year_folder / f"shp{yy}_p_user.sas7bdat"
    household_file = year_folder / f"shp{yy}_h_user.sas7bdat"

    # Return both paths for now
    return person_file, household_file


# Test the function using 2004
test_person, test_household = get_year_files(2004)

# Check that both files exist
print("Individual file exists:", test_person.exists())
print("Household file exists:", test_household.exists())

# Display the file names
print("\nIndividual file:", test_person.name)
print("Household file:", test_household.name)

In [ ]:

# Define a function to prepare data for one survey year
def prepare_year(year):

    # Calculate the SHP wave number
    wave = year - 1998

    # Get the last two digits of the year
    yy = str(year)[-2:]

    # Build the folder path for the selected year
    year_folder = (
        Path.home()
        / "Documents"
        / "swissubase_932_12_0"
        / "Data_SAS"
        / "SHP-Data-W1-W26-SAS"
        / f"W{wave}_{year}"
    )

    # Build the individual and household file paths
    person_file = year_folder / f"shp{yy}_p_user.sas7bdat"
    household_file = year_folder / f"shp{yy}_h_user.sas7bdat"

    # Select the individual variables needed for our research
    person_variables = [
        "IDPERS",
        f"IDHOUS{yy}",
        f"AGE{yy}",
        f"SEX{yy}",
        f"OCCUPA{yy}",
        f"WSTAT{yy}",
        f"IS1MAJ{yy}",
        f"P{yy}W228"
    ]

    # Select the household variables
    household_variables = [
        f"IDHOUS{yy}",
        f"REGION{yy}"
    ]

    # Load only the selected individual variables
    person_data, _ = pyreadstat.read_sas7bdat(
        str(person_file),
        usecols=person_variables
    )

    # Load only the selected household variables
    household_data, _ = pyreadstat.read_sas7bdat(
        str(household_file),
        usecols=household_variables
    )

    # Create copies to preserve the loaded datasets
    person_clean = person_data.copy()
    household_clean = household_data.copy()

    # Replace invalid ages with NaN
    person_clean.loc[
        ~person_clean[f"AGE{yy}"].between(0, 120),
        f"AGE{yy}"
    ] = float("nan")

    # Keep only valid sex codes
    person_clean.loc[
        ~person_clean[f"SEX{yy}"].isin([1, 2, 3]),
        f"SEX{yy}"
    ] = float("nan")

    # Keep only valid occupation-from-grid codes
    person_clean.loc[
        ~person_clean[f"OCCUPA{yy}"].between(1, 11),
        f"OCCUPA{yy}"
    ] = float("nan")

    # Keep only valid working status codes
    person_clean.loc[
        ~person_clean[f"WSTAT{yy}"].isin([1, 2, 3]),
        f"WSTAT{yy}"
    ] = float("nan")

    # Keep only valid occupation codes
    person_clean.loc[
        ~person_clean[f"IS1MAJ{yy}"].between(0, 9),
        f"IS1MAJ{yy}"
    ] = float("nan")

    # Keep only whole-number satisfaction scores from 0 to 10
    satisfaction = person_clean[f"P{yy}W228"]

    valid_satisfaction = (
        satisfaction.between(0, 10)
        & satisfaction.eq(satisfaction.round())
    )

    person_clean.loc[
        ~valid_satisfaction,
        f"P{yy}W228"
    ] = float("nan")

    # Keep only valid region codes
    household_clean.loc[
        ~household_clean[f"REGION{yy}"].between(1, 7),
        f"REGION{yy}"
    ] = float("nan")

    # Merge individuals with their household information
    merged_data = person_clean.merge(
        household_clean,
        on=f"IDHOUS{yy}",
        how="left",
        validate="many_to_one",
        indicator=True
    )

    # Count respondents who matched a household
    matched = (merged_data["_merge"] == "both").sum()

    # Check for unmatched respondents
    if matched != len(person_clean):
        print(f"Warning: {year} has unmatched respondents")

    # Remove the temporary merge-check column
    merged_data = merged_data.drop(columns=["_merge"])

    # Rename the variables to consistent names
    standardised_data = merged_data.rename(
        columns={
            f"IDHOUS{yy}": "IDHOUS",
            f"AGE{yy}": "AGE",
            f"SEX{yy}": "SEX",
            f"OCCUPA{yy}": "EMPLOYMENT_STATUS",
            f"WSTAT{yy}": "WORKING_STATUS",
            f"IS1MAJ{yy}": "OCCUPATION",
            f"P{yy}W228": "JOB_SATISFACTION",
            f"REGION{yy}": "REGION"
        }
    )

    # Add the survey year
    standardised_data["YEAR"] = year

    # Arrange the columns in a consistent order
    standardised_data = standardised_data[
        [
            "IDPERS",
            "IDHOUS",
            "YEAR",
            "AGE",
            "SEX",
            "EMPLOYMENT_STATUS",
            "WORKING_STATUS",
            "OCCUPATION",
            "JOB_SATISFACTION",
            "REGION"
        ]
    ]

    # Return the cleaned, merged and standardised dataset
    return standardised_data


In [ ]:
# Test the complete preparation function using 2004
test_standardised = prepare_year(2004)

# Check the dataset dimensions
print("Dataset shape:", test_standardised.shape)

# Display the column names
print("\nColumns:")
print(test_standardised.columns.tolist())

# Check respondent IDs
print(
    "\nDuplicate respondent IDs:",
    test_standardised["IDPERS"].duplicated().sum()
)

# Compare the automated result with our manually prepared data
print(
    "\nMatches manually prepared 2004 dataset:",
    test_standardised.equals(standardised_2004)
)

# Preview the data
display(test_standardised.head())

### 5.4 Validating the Automated Preparation Process

#### Purpose

Before applying the automated function to all 21 survey years, we test it using the 2024 dataset.

We previously prepared the 2024 data manually in Sections 2–4. We can therefore compare those results with the dataset produced by our automated function.

#### Validation Steps

We check:

1. Whether the function successfully loads and prepares the 2024 data.
2. Whether the automated dataset contains the expected variables.
3. Whether the number of observations matches the manually prepared dataset.
4. Whether the cleaned values and merged household information are consistent.
5. Whether the dataset contains any unexpected duplicate records.

#### Objective

This validation helps us identify errors in the automated preparation process before applying it to the remaining survey years.

If the results are consistent, we can use the function to prepare the full 2004–2024 dataset.

In [ ]:
# Prepare the 2024 dataset using our automated function
test_2024 = prepare_year(2024)

# Check the dataset dimensions
print("Automated 2024 shape:", test_2024.shape)

# Check the manually prepared 2024 dataset
print("Manual 2024 shape:", person_household_2024.shape)

# Compare the column names
print(
    "\nSame columns:",
    set(test_2024.columns) == set(person_household_2024.columns)
)

# Check that working status matches the manually prepared data
automated_status = test_2024.sort_values("IDPERS")["WORKING_STATUS"].reset_index(drop=True)

manual_status = person_household_2024.sort_values("IDPERS")["WORKING_STATUS"].reset_index(drop=True)

print(
    "Working status matches manual data:",
    automated_status.equals(manual_status)
)

# Compare the number of respondents
print(
    "Same number of respondents:",
    len(test_2024) == len(person_household_2024)
)

# Check for duplicate respondent IDs
print(
    "\nDuplicate respondent IDs:",
    test_2024["IDPERS"].duplicated().sum()
)

# Check missing values in the automated dataset
print("\nMissing values:")
print(test_2024.isna().sum())

# Preview the first five rows
display(test_2024.head())

In [ ]:
# Arrange the manual dataset in the same column order
manual_2024 = person_household_2024[
    test_2024.columns
].copy()

# Sort both datasets by respondent ID
automated_sorted = (
    test_2024
    .sort_values("IDPERS")
    .reset_index(drop=True)
)

manual_sorted = (
    manual_2024
    .sort_values("IDPERS")
    .reset_index(drop=True)
)

# Compare the respondent IDs
print(
    "Same respondent IDs:",
    automated_sorted["IDPERS"].equals(
        manual_sorted["IDPERS"]
    )
)

# Compare all values, treating matching NaN values as equal
comparison = automated_sorted.eq(manual_sorted) | (
    automated_sorted.isna() & manual_sorted.isna()
)

# Count differences in each column
differences = (~comparison).sum()

print("\nDifferences by column:")
print(differences)

# Check whether all values match
print(
    "\nAll values match:",
    comparison.to_numpy().all()
)


#### Interpretation — Validation of the Automated Preparation Process

The automated preparation function successfully reproduced the manually prepared 2024 dataset.

The validation results show that:

- Both datasets contain **14,582 observations and 10 variables**.
- Both datasets have the same respondent identifiers.
- No duplicate respondent identifiers were found.
- All 10 variables matched between the manual and automated datasets.
- No differences were found in any column, including missing values.

The final comparison returned `All values match: True`, confirming that the automated function produced exactly the same results as the manual preparation for 2024.

This gives us confidence that the function works correctly for the tested year and can be applied to the remaining survey years (2004–2024).

However, matching results for 2024 do not guarantee that all variables are fully comparable across different survey years. This will need to be considered when analysing the combined dataset.


### 5.5 Combining the Annual Datasets (2004–2024)

#### Purpose

After testing the automated preparation function, we apply it to all 21 survey years from 2004 to 2024.

The goal is to combine the individual annual datasets into one dataset that can be used to analyse job satisfaction over time.

#### Combining the Data

We use a loop to:

1. Process each survey year using `prepare_year(year)`.
2. Store the prepared annual datasets in a list.
3. Combine the datasets into one DataFrame using `pd.concat()`.
4. Check the number of observations and variables in the combined dataset.
5. Verify that all 21 survey years are included.

Each row represents one respondent in one survey year.

#### Longitudinal Structure

The SHP is a longitudinal survey, meaning that some respondents participate in multiple years.

We use:
- `IDPERS` to identify respondents across survey years.
- `YEAR` to identify the survey year.
- `IDHOUS` to identify household information.

The same respondent can appear in several years, but each respondent should appear only once per survey year.

#### Data Validation

After combining the datasets, we check:

- Total number of observations.
- Number of unique respondents.
- Number of observations per year.
- Missing values in the selected variables.
- Duplicate combinations of `IDPERS` and `YEAR`.

These checks help confirm that the combined dataset is ready for sample selection and descriptive analysis.

In [ ]:
# Define all 21 survey years from 2004 to 2024
years = range(2004, 2025)

# Create an empty list to store the prepared yearly datasets
yearly_datasets = []

# Process each survey year from 2004 to 2024
for year in years:

    # Load, clean, merge and standardise this year's data
    yearly_data = prepare_year(year)

    # Add the prepared dataset to our list
    yearly_datasets.append(yearly_data)

    # Display progress and the number of respondents
    print(f"{year}: {len(yearly_data)} respondents prepared")

# Combine all yearly datasets into one dataset
full_dataset = pd.concat(
    yearly_datasets,
    ignore_index=True
)

# Display the combined dataset dimensions
print("\nCombined dataset shape:", full_dataset.shape)

# Check how many survey years are included
print("Number of survey years:", full_dataset["YEAR"].nunique())

# Display the first five rows
display(full_dataset.head())

In [ ]:
# Check the combined dataset dimensions
print("Combined dataset shape:", full_dataset.shape)

# Check the number of survey years
print("\nNumber of survey years:", full_dataset["YEAR"].nunique())

# Check the number of observations in each year
print("\nObservations by year:")
print(
    full_dataset["YEAR"]
    .value_counts()
    .sort_index()
)

# Check duplicate respondent-year combinations
duplicate_person_year = full_dataset.duplicated(
    subset=["IDPERS", "YEAR"]
).sum()

print("\nDuplicate respondent-year records:", duplicate_person_year)

# Check missing respondent IDs and survey years
print("Missing respondent IDs:", full_dataset["IDPERS"].isna().sum())
print("Missing survey years:", full_dataset["YEAR"].isna().sum())

# Check missing values across all variables
print("\nMissing values by variable:")
print(full_dataset.isna().sum())

# Check the number of unique respondents
print(
    "\nUnique respondents across all years:",
    full_dataset["IDPERS"].nunique()
)

#### Interpretation — Combined Dataset Validation (2004–2024)

The automated preparation process successfully combined all 21 survey years into one dataset.

The main results were:

- The combined dataset contains **305,462 observations and 10 variables**.
- All 21 survey years from 2004 to 2024 are included.
- There are **45,632 unique respondents** across the period.
- No duplicate respondent-year combinations were found.
- No respondent IDs or survey years are missing.
- Household IDs and region values are also complete.

Some variables contain missing values:

- Age: 232 missing values.
- Sex: 4 missing values.
- Employment status: 1,003 missing values.
- Occupation: 180,252 missing values.
- Job satisfaction: 178,161 missing values.

The large number of missing occupation and job satisfaction values is partly expected because the full dataset includes respondents who are not employed or were not asked employment-related questions.

The dataset is longitudinal, meaning the same respondent can appear in multiple survey years. Therefore, the 305,462 observations do not represent 305,462 different individuals.

Overall, the combined dataset passed the main structural checks and is ready for selecting the employed analysis sample.

### 5.6 Defining the Employed Analysis Sample

#### Purpose

The combined dataset includes both employed and non-employed respondents. Since our research focuses on job satisfaction across occupations, we select employed respondents with valid occupation and job satisfaction information.

#### Sample Selection

We apply three criteria:

1. **Working status:** Keep respondents with `WORKING_STATUS == 1` (employed).
2. **Occupation:** Keep respondents with a valid ISCO major occupational group (0–9).
3. **Job satisfaction:** Keep respondents with a valid satisfaction score between 0 and 10.

`EMPLOYMENT_STATUS` is retained as an additional variable but is not used to determine whether a respondent is employed.

#### Sample Validation

We check the number of observations, unique respondents, survey years, missing values and duplicate respondent-year records.

#### Employed Sample

| Measure | Result |
|---|---:|
| Survey period | 2004–2024 |
| Survey years | 21 |
| Respondent-year observations | 124,769 |
| Unique respondents | 22,265 |
| Duplicate respondent-year records | 0 |

Some respondents appear in multiple survey years because the SHP is a longitudinal survey.

#### Limitations

Respondents with missing occupation or job satisfaction information are excluded, which may affect representativeness.

The dataset contains repeated observations from the same individuals. This must be considered when conducting statistical tests.

Survey weights have not yet been applied, so the descriptive results are unweighted.

In [ ]:

# Identify respondents with valid occupation and satisfaction
has_both = (
    full_dataset["OCCUPATION"].notna()
    & full_dataset["JOB_SATISFACTION"].notna()
)

# Create a sample with both research variables available
available_sample = full_dataset.loc[
    has_both
].copy()

# Keep respondents classified as employed using WSTAT
# WORKING_STATUS = 1 represents employed respondents
employed_sample = available_sample.loc[
    available_sample["WORKING_STATUS"] == 1
].copy()

# Display sample sizes
print("Total observations:", len(full_dataset))
print("Valid occupation and satisfaction:", len(available_sample))
print("Employed analysis observations:", len(employed_sample))

# Check the number of survey years
print("\nSurvey years:", employed_sample["YEAR"].nunique())

# Check the number of unique respondents
print("Unique employed respondents:", employed_sample["IDPERS"].nunique())

# Check for duplicate respondent-year combinations
print(
    "Duplicate respondent-year records:",
    employed_sample.duplicated(subset=["IDPERS", "YEAR"]).sum()
)

# Check missing values in the analysis sample
print("\nMissing values:")
print(employed_sample.isna().sum())

# Display the number of employed observations per year
print("\nEmployed observations by year:")
print(employed_sample["YEAR"].value_counts().sort_index())

In [ ]:

# Check the distribution of working status in the full dataset
print("Working status in the full dataset:")
print(
    full_dataset["WORKING_STATUS"]
    .value_counts(dropna=False)
    .sort_index()
)

# Check working status among respondents with valid research variables
print("\nWorking status with valid occupation and satisfaction:")
print(
    available_sample["WORKING_STATUS"]
    .value_counts(dropna=False)
    .sort_index()
)


In [ ]:

# Identify employed observations with missing age or sex
missing_demographics = employed_sample.loc[
    employed_sample["AGE"].isna()
    | employed_sample["SEX"].isna()
]

# Display the affected observations
print("Observations with missing age or sex:", len(missing_demographics))
display(missing_demographics)


In [ ]:

# Create a sample with complete demographic information
# Keep the original employed_sample unchanged
analysis_sample = employed_sample.dropna(
    subset=["AGE", "SEX"]
).copy()

# Compare the sample sizes
print("Original employed sample:", len(employed_sample))
print("Complete demographic sample:", len(analysis_sample))
print("Observations excluded:", len(employed_sample) - len(analysis_sample))

# Check that no age or sex values are missing
print("\nMissing demographic values:")
print(analysis_sample[["AGE", "SEX"]].isna().sum())



#### Interpretation — Employed Analysis Sample (2004–2024)

After filtering the combined dataset, we obtained the final employed sample for analysing job satisfaction across occupations and survey years.

The main results were:

- The employed sample contains **124,769 respondent-year observations**.
- These observations represent **22,265 unique respondents**.
- All 21 survey years from 2004 to 2024 are included.
- No duplicate respondent-year combinations were found.
- All selected observations have valid occupation and job satisfaction values.
- Only 2 observations have missing age values, and 1 has a missing sex value.
- The employed sample contains 5,421 observations in 2004 and 6,007 in 2024.

The sample was created by selecting respondents with `WORKING_STATUS == 1` and valid occupation and job satisfaction information. The preliminary `EMPLOYMENT_STATUS` filter used in Section 4 was not used for the final sample.

We also created `analysis_sample` by excluding the 3 observations with missing age or sex information. This gives us **124,766 observations** for analyses requiring complete demographic information.

Because the SHP is a longitudinal survey, respondents can appear in multiple survey years. Therefore, respondent-year observations are not necessarily independent.

The number and composition of observations also vary between years, which should be considered when interpreting changes in job satisfaction.

Overall, the employed sample is ready for descriptive analysis by year, occupation, sex, age and region of residence.

The results are currently unweighted. Survey weights and repeated observations will need to be considered before conducting formal statistical tests or making population-level conclusions.


### 5.7 Composition of the Analysis Sample

#### Purpose

We examine the composition of the employed analysis sample across four characteristics:

1. Occupational group (ISCO major groups).
2. Age group.
3. Sex.
4. Region of residence.

#### Analysis

For each characteristic, we calculate:

- The number of respondent-year observations.
- The percentage of observations belonging to each category.
- A bar chart showing the distribution across categories.

This helps us identify which groups are more or less represented in the dataset before comparing their job satisfaction levels.

#### Limitations

The analysis is descriptive and unweighted. Respondents may appear in multiple survey years, so the counts represent observations rather than unique individuals.

Differences in group sizes may affect the precision of later comparisons.

In [ ]:
# 5.7 Composition of the Analysis Sample (2004–2024)

import pandas as pd
import matplotlib.pyplot as plt

# 1. Select the analysis sample
sample = analysis_sample.copy()

print("Total respondent-year observations:", len(sample))

# 2. Define age groups
age_bins = [0, 24, 34, 44, 54, 64, float("inf")]
age_labels = ["Under 25", "25–34", "35–44", "45–54", "55–64", "65+"]

sample["AGE_GROUP"] = pd.cut(
    sample["AGE"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

# 3. Define readable occupational group labels
occupation_labels = {
    0: "Armed forces",
    1: "Managers",
    2: "Professionals",
    3: "Technicians and associate professionals",
    4: "Clerical support workers",
    5: "Service and sales workers",
    6: "Skilled agricultural workers",
    7: "Craft and related trades workers",
    8: "Plant and machine operators",
    9: "Elementary occupations"
}

sample["OCCUPATION_LABEL"] = sample["OCCUPATION"].map(
    occupation_labels
)

# 4. Define readable sex labels
sex_labels = {
    1: "Men",
    2: "Women",
    3: "Other"
}

sample["SEX_LABEL"] = sample["SEX"].map(
    sex_labels
)

# 5. Define readable regional labels
region_labels = {
    1: "Lake Geneva",
    2: "Espace Mittelland",
    3: "Northwestern Switzerland",
    4: "Zurich",
    5: "Eastern Switzerland",
    6: "Central Switzerland",
    7: "Ticino"
}

sample["REGION_LABEL"] = sample["REGION"].map(
    region_labels
)

# 6. Define variables for sample composition
variables = {
    "OCCUPATION_LABEL": "Occupational group (ISCO)",
    "AGE_GROUP": "Age group",
    "SEX_LABEL": "Sex",
    "REGION_LABEL": "Region of residence"
}

# 7. Calculate observation counts and percentages
for column, title in variables.items():

    counts = sample[column].value_counts(dropna=False)

    percentages = (
        sample[column]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
    )

    summary = pd.DataFrame({
        "Observations": counts,
        "Percentage (%)": percentages
    })

    # Keep age groups in chronological order
    if column == "AGE_GROUP":
        summary = summary.sort_index()
    else:
        summary = summary.sort_values(
            "Observations",
            ascending=False
        )

    print(f"\n{title}")
    display(summary.round(2))

# 8. Create four charts showing sample composition
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

for ax, (column, title) in zip(axes.flat, variables.items()):

    chart_data = sample[column]

    # Exclude the single "Other" observation from the sex chart only
    if column == "SEX_LABEL":
        chart_data = chart_data[
            chart_data.isin(["Men", "Women"])
        ]

    counts = chart_data.value_counts()

    # Keep age groups in chronological order
    if column == "AGE_GROUP":
        counts = counts.sort_index()
    else:
        counts = counts.sort_values(ascending=False)

    counts.plot.bar(
        ax=ax,
        color="steelblue"
    )

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Number of observations")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)

# 9. Finalise chart layout
plt.suptitle(
    "Composition of the Analysis Sample (2004–2024)",
    fontsize=14
)

plt.tight_layout()
plt.show()

#### Results and Interpretation

The analysis sample contains 124,766 respondent-year observations between 2004 and 2024.

**1. Occupational Composition**

- Professionals (ISCO 2) are the largest occupational group, with 37,981 observations (30.44%).
- Technicians and associate professionals (ISCO 3) account for 21,855 observations (17.52%).
- Clerical support workers (ISCO 4) account for 17,013 observations (13.64%).
- Service and sales workers (ISCO 5) account for 16,173 observations (12.96%).
- Armed forces occupations (ISCO 0) have only 131 observations (0.10%).

The occupational groups are unevenly represented, which is important when comparing job satisfaction, particularly for smaller groups.

**2. Age Composition**

- Respondents aged 45–54 represent the largest age group, with 32,967 observations (26.42%).
- Those aged 55–64 account for 26,224 observations (21.02%).
- Respondents aged 35–44 account for 24,653 observations (19.76%).
- The smallest group is respondents aged 65 and above, with 6,926 observations (5.55%).

The sample contains more observations from middle-aged respondents than from the youngest and oldest age groups.

**3. Sex Composition**

- Women account for 63,958 observations (51.26%).
- Men account for 60,807 observations (48.74%).
- The category "Other" appears once, representing less than 0.01% of observations.

Men and women are represented in relatively similar proportions.

**4. Regional Composition**

- Espace Mittelland has the largest share, with 31,604 observations (25.33%).
- Lake Geneva accounts for 21,562 observations (17.28%).
- Zurich accounts for 21,110 observations (16.92%).
- Northwestern Switzerland accounts for 17,396 observations (13.94%).
- Eastern Switzerland accounts for 15,906 observations (12.75%).
- Central Switzerland accounts for 13,002 observations (10.42%).
- Ticino has the smallest share, with 4,186 observations (3.36%).

Regional sample sizes vary substantially, which should be considered when interpreting regional comparisons.

**5. Implications for Further Analysis**

- Sample sizes differ across occupations, age groups and regions.
- Smaller groups may produce less precise estimates of average job satisfaction.
- The sample composition may change between survey years, affecting comparisons over time.
- These figures represent unweighted respondent-year observations rather than unique individuals or population proportions.

**Conclusion**

The analysis sample includes observations across all major occupational groups, age groups, sex categories and Swiss regions. However, the groups are not equally represented. We will consider these differences when interpreting job satisfaction patterns and economic-event comparisons.

In [ ]:
# Check individual survey-weight availability for 2004–2024

weight_check = []

for year in range(2004, 2025):
    wave = year - 1998
    filename = data_folder / f"W{wave}_{year}" / f"shp{str(year)[-2:]}_p_user.sas7bdat"

    expected_weight = f"WI{year % 100:02d}CSS"

    if not filename.is_file():
        weight_check.append({
            "Year": year,
            "Expected_weight": expected_weight,
            "Available": False,
            "Other_weight_columns": "File not found"
        })
        continue

    _, meta = pyreadstat.read_sas7bdat(
        str(filename),
        metadataonly=True
    )

    columns = [c.upper() for c in meta.column_names]
    other_weights = [
        c for c in columns
        if c.startswith("WI") and ("CS" in c or "LS" in c)
    ]

    weight_check.append({
        "Year": year,
        "Expected_weight": expected_weight,
        "Available": expected_weight in columns,
        "Other_weight_columns": ", ".join(other_weights)
    })

weight_check_df = pd.DataFrame(weight_check)

display(weight_check_df)

print(
    "Years with expected weight:",
    weight_check_df["Available"].sum(),
    "/ 21"
)

In [ ]:
# Check annual individual cross-sectional weight quality

weight_quality = []

for year in range(2004, 2025):
    wave = year - 1998
    filename = (
        data_folder
        / f"W{wave}_{year}"
        / f"shp{year % 100:02d}_p_user.sas7bdat"
    )

    weight_name = f"WI{year % 100:02d}CSS"

    weights_df, _ = pyreadstat.read_sas7bdat(
        str(filename),
        usecols=[weight_name]
    )

    weights = pd.to_numeric(
        weights_df[weight_name],
        errors="coerce"
    )

    weight_quality.append({
        "Year": year,
        "Observations": len(weights),
        "Valid_weights": int((weights > 0).sum()),
        "Missing": int(weights.isna().sum()),
        "Zero_or_negative": int(
            ((weights <= 0) & weights.notna()).sum()
        ),
        "Minimum_positive": weights[weights > 0].min(),
        "Maximum": weights.max(),
        "Mean": weights.mean()
    })

weight_quality_df = pd.DataFrame(weight_quality)

display(weight_quality_df.round(2))

print(
    "\nTotal years checked:",
    len(weight_quality_df)
)

In [ ]:
# Investigate zero and negative survey-weight values

weight_codes = []

for year in range(2004, 2025):
    wave = year - 1998

    filename = (
        data_folder
        / f"W{wave}_{year}"
        / f"shp{year % 100:02d}_p_user.sas7bdat"
    )

    weight_name = f"WI{year % 100:02d}CSS"

    df, _ = pyreadstat.read_sas7bdat(
        str(filename),
        usecols=[weight_name]
    )

    weights = pd.to_numeric(df[weight_name], errors="coerce")

    nonpositive = weights[weights <= 0]

    counts = nonpositive.value_counts().sort_index()

    for value, count in counts.items():
        weight_codes.append({
            "Year": year,
            "Weight_value": value,
            "Count": count
        })

weight_codes_df = pd.DataFrame(weight_codes)

display(weight_codes_df)

print("\nSummary by weight value:")
display(
    weight_codes_df.groupby("Weight_value")["Count"]
    .sum()
    .reset_index()
)

In [ ]:
# Check survey-weight coverage in the employed sample

weight_parts = []

for year in range(2004, 2025):
    wave = year - 1998

    filename = (
        data_folder
        / f"W{wave}_{year}"
        / f"shp{year % 100:02d}_p_user.sas7bdat"
    )

    weight_name = f"WI{year % 100:02d}CSS"

    df, _ = pyreadstat.read_sas7bdat(
        str(filename),
        usecols=["IDPERS", weight_name]
    )

    df = df.rename(columns={weight_name: "WEIGHT"})
    df["YEAR"] = year

    weight_parts.append(df)

all_weights = pd.concat(weight_parts, ignore_index=True)

# Ensure each respondent-year has only one weight
assert not all_weights.duplicated(["IDPERS", "YEAR"]).any()

# Merge without modifying the original employed_sample
employed_weight_check = employed_sample.merge(
    all_weights,
    on=["IDPERS", "YEAR"],
    how="left",
    validate="many_to_one"
)

# Check coverage
weight_coverage = (
    employed_weight_check
    .groupby("YEAR")
    .agg(
        Observations=("WEIGHT", "size"),
        Positive_weights=("WEIGHT", lambda x: (x > 0).sum()),
        Zero_weights=("WEIGHT", lambda x: (x == 0).sum()),
        Negative_codes=("WEIGHT", lambda x: (x < 0).sum()),
        Missing_weights=("WEIGHT", lambda x: x.isna().sum())
    )
    .reset_index()
)

weight_coverage["Coverage_%"] = (
    weight_coverage["Positive_weights"]
    / weight_coverage["Observations"] * 100
).round(2)

display(weight_coverage)

print(
    "\nTotal employed observations:",
    len(employed_weight_check)
)

print(
    "Observations with positive weights:",
    (employed_weight_check["WEIGHT"] > 0).sum()
)

print(
    "Overall coverage (%):",
    round(
        (employed_weight_check["WEIGHT"] > 0).mean() * 100,
        2
    )
)


### 5.8 Survey Weight Verification

The SHP provides individual cross-sectional survey weights for all 21 years (2004–2024).

We verified the availability and quality of these variables and matched them to the final employed sample using respondent ID and survey year.

Of the 124,769 employed observations, 121,547 (97.42%) have positive survey weights.

The remaining 3,222 observations have zero or negative weight codes and are excluded from weighted calculations.

The following comparison examines whether weighted and unweighted annual job satisfaction averages differ. The original descriptive analysis remains unchanged.


In [ ]:
# Compare weighted and unweighted annual job satisfaction

comparison_data = employed_weight_check.copy()

comparison_data["VALID_WEIGHT"] = comparison_data["WEIGHT"].where(
    comparison_data["WEIGHT"] > 0
)

annual_comparison = []

for year, group in comparison_data.groupby("YEAR"):

    valid = group.dropna(
        subset=["JOB_SATISFACTION", "VALID_WEIGHT"]
    )

    unweighted_mean = group["JOB_SATISFACTION"].mean()

    weighted_mean = np.average(
        valid["JOB_SATISFACTION"],
        weights=valid["VALID_WEIGHT"]
    ) if len(valid) > 0 else np.nan

    annual_comparison.append({
        "Year": year,
        "Unweighted_mean": unweighted_mean,
        "Weighted_mean": weighted_mean,
        "Difference": weighted_mean - unweighted_mean,
        "Weighted_observations": len(valid)
    })

annual_comparison_df = pd.DataFrame(annual_comparison)

display(annual_comparison_df.round(3))

print(
    "\nMaximum absolute difference:",
    round(annual_comparison_df["Difference"].abs().max(), 3)
)


#### Interpretation — Survey Weight Verification

The Swiss Household Panel (SHP) provides individual cross-sectional survey weights for all 21 years (2004–2024). We verified the availability of these variables and matched them to the employed sample using respondent ID and survey year.

Of the 124,769 employed observations, 121,547 (97.42%) have positive survey weights. The remaining 3,222 observations have zero or negative weight codes and were excluded from weighted calculations.

**Weighted vs. unweighted job satisfaction:**

| Measure | 2004 | 2024 | Change |
|---|---:|---:|---:|
| Unweighted mean | 8.212 | 7.815 | −0.397 |
| Weighted mean | 8.207 | 7.711 | −0.496 |

The weighted results confirm the overall downward trend in job satisfaction. Both methods show a decline between 2004 and 2024, although the weighted decline is slightly larger.

Weighted averages were lower than unweighted averages in all 21 years. The largest difference was approximately 0.156 points in 2021.

**Limitations:**

- Individual cross-sectional weights were applied to the employed subgroup without additional subgroup-specific adjustments.
- Observations with nonpositive weights were excluded from weighted calculations.
- The comparison does not account for all aspects of the SHP's complex survey design.
- Repeated observations of the same respondents are not explicitly modelled in this descriptive comparison.
- The weighted results are used as a robustness comparison rather than replacing the original descriptive analysis.

Overall, the survey-weight comparison supports the main finding that average job satisfaction declined between 2004 and 2024.


## 6. Exploratory Descriptive Analysis (EDA)

### 6.1 Overall Job Satisfaction (2004–2024)

#### Purpose

This section examines overall job satisfaction among employed respondents in Switzerland between 2004 and 2024.

We first examine the distribution of job satisfaction scores across the full period. We then analyse how average satisfaction changes over time, including around major economic events.

The analysis uses the employed sample and is descriptive and unweighted. Respondents may appear in multiple survey years.

---

#### 6.1.1 Distribution of Job Satisfaction

##### Purpose

We examine how job satisfaction scores are distributed across all observations from 2004 to 2024.

Job satisfaction is measured on a scale from 0 (not at all satisfied) to 10 (completely satisfied).

##### Analysis

We calculate:

1. **Descriptive statistics:** Number of observations, mean, median, standard deviation, minimum and maximum.
2. **High satisfaction:** Percentage of observations reporting a score between 8 and 10.
3. **Low satisfaction:** Percentage of observations reporting a score between 0 and 5.
4. **Score distribution:** Percentage of observations reporting each score from 0 to 10.
5. **Visualisation:** Bar chart showing the frequency of each satisfaction score.

##### Objective

This analysis helps us understand the overall distribution of job satisfaction before examining differences across years and population groups.


In [ ]:
# 6.1.1 Distribution of Job Satisfaction (2004–2024)

# Select valid job satisfaction scores
satisfaction = employed_sample["JOB_SATISFACTION"].dropna()

# 1. Descriptive statistics
print("Overall job satisfaction — descriptive statistics:")
display(satisfaction.describe().round(2))

# 2. High and low satisfaction percentages
high_satisfaction = (satisfaction >= 8).mean() * 100
low_satisfaction = (satisfaction <= 5).mean() * 100

print(f"\nHigh satisfaction (8–10): {high_satisfaction:.1f}%")
print(f"Low satisfaction (0–5): {low_satisfaction:.1f}%")

# 3. Percentage of observations for each satisfaction score
satisfaction_distribution = (
    satisfaction.value_counts(normalize=True)
    .reindex(range(11), fill_value=0)
    .sort_index()
    .mul(100)
)

print("\nDistribution of satisfaction scores (%):")
display(
    satisfaction_distribution
    .rename("Percentage (%)")
    .round(2)
    .to_frame()
)

# 4. Bar chart
plt.figure(figsize=(10, 5))

plt.bar(
    satisfaction_distribution.index,
    satisfaction_distribution.values
)

plt.title("Distribution of Job Satisfaction (2004–2024)")
plt.xlabel("Job satisfaction score (0–10)")
plt.ylabel("Percentage of observations")
plt.xticks(range(11))
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

##### Results and Interpretation

The distribution analysis includes 124,769 observations from employed respondents between 2004 and 2024.

**1. Overall Job Satisfaction**

- Mean job satisfaction is 7.97 out of 10, with a median of 8.
- The standard deviation is 1.48, indicating variation in satisfaction scores.
- The middle 50% of observations fall between 7 and 9.
- The observed scores range from 0 to 10.

**2. High and Low Satisfaction**

- 69.8% of observations report high job satisfaction (8–10).
- Only 6.5% report low job satisfaction (0–5).
- The remaining 23.7% report scores of 6 or 7.

**3. Distribution of Satisfaction Scores**

- A score of 8 is the most common response, representing 35.95% of observations.
- Scores of 7 (18.04%), 9 (17.58%) and 10 (16.25%) are also common.
- Scores below 5 are relatively uncommon, with each individual score representing less than 1.3% of observations.
- The distribution is concentrated towards higher satisfaction scores, with a longer tail towards lower scores (negative skewness).

**4. Implications for Further Analysis**

- The concentration of scores between 7 and 10 suggests that average satisfaction alone may not capture all differences between groups.
- We therefore complement mean satisfaction with the percentage of observations reporting low satisfaction (0–5) in subsequent analyses.
- Since satisfaction is already relatively high, differences in average scores between occupational and demographic groups may appear small.
- These results are descriptive and unweighted. Respondents may contribute observations in multiple survey years.

**Conclusion**

Overall, employed respondents report relatively high job satisfaction between 2004 and 2024. However, the distribution also shows a smaller group reporting low satisfaction. Examining both average satisfaction and the share of low satisfaction will provide a more complete comparison across occupations, demographic groups and economic periods.

In [ ]:
# Group employed respondents by survey year
yearly_satisfaction = (
    employed_sample
    .groupby("YEAR")["JOB_SATISFACTION"]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

# Rename the columns to make the table easier to read
yearly_satisfaction = yearly_satisfaction.rename(
    columns={
        "count": "Respondents",
        "mean": "Mean satisfaction",
        "median": "Median satisfaction",
        "std": "Standard deviation"
    }
)

# Display the yearly results
print("Job satisfaction by survey year (2004–2024):")
print(yearly_satisfaction.to_string())

# Identify the years with the highest and lowest averages
highest_year = yearly_satisfaction["Mean satisfaction"].idxmax()
lowest_year = yearly_satisfaction["Mean satisfaction"].idxmin()

print("\nHighest average satisfaction:", highest_year)
print("Lowest average satisfaction:", lowest_year)

# Compare average job satisfaction in 2004 and 2024
mean_2004 = yearly_satisfaction.loc[2004, "Mean satisfaction"]
mean_2024 = yearly_satisfaction.loc[2024, "Mean satisfaction"]

change = round(mean_2024 - mean_2004, 2)

print("\nMean satisfaction in 2004:", mean_2004)
print("Mean satisfaction in 2024:", mean_2024)
print("Change from 2004 to 2024:", change)

In [ ]:
# Import the plotting library
import matplotlib.pyplot as plt

# Create the figure
plt.figure(figsize=(12, 5))

# Plot average job satisfaction by year
plt.plot(
    yearly_satisfaction.index,
    yearly_satisfaction["Mean satisfaction"],
    marker="o",
    linewidth=2
)

# Add a descriptive chart title
plt.title("Average Job Satisfaction in Switzerland (2004–2024)")

# Label the horizontal and vertical axes
plt.xlabel("Survey year")
plt.ylabel("Mean job satisfaction (0–10)")

# Display every survey year on the horizontal axis
plt.xticks(range(2004, 2025), rotation=45)

# Set the vertical axis to the full satisfaction scale
plt.ylim(0, 10)

# Add a grid to make values easier to read
plt.grid(True, alpha=0.3)

# Adjust spacing so labels fit
plt.tight_layout()

# Display the chart
plt.show()

In [ ]:
# Create a new figure
plt.figure(figsize=(12, 5))

# Plot average job satisfaction by year
plt.plot(
    yearly_satisfaction.index,
    yearly_satisfaction["Mean satisfaction"],
    marker="o",
    linewidth=2,
    label="Average job satisfaction"
)

# Mark the 2009 financial crisis
plt.axvline(
    x=2009,
    color="red",
    linestyle="--",
    alpha=0.7,
    label="2009 financial crisis"
)

# Mark the 2015 Swiss franc shock
plt.axvline(
    x=2015,
    color="orange",
    linestyle="--",
    alpha=0.7,
    label="2015 Swiss franc shock"
)

# Mark the 2020 COVID-19 pandemic
plt.axvline(
    x=2020,
    color="purple",
    linestyle="--",
    alpha=0.7,
    label="2020 COVID-19"
)

# Add the chart title and axis labels
plt.title("Job Satisfaction in Switzerland and Major Economic Events (2004–2024)")
plt.xlabel("Survey year")
plt.ylabel("Mean job satisfaction (0–10)")

# Display every survey year
plt.xticks(range(2004, 2025), rotation=45)

# Keep the full satisfaction scale
plt.ylim(0, 10)

# Add a legend and grid
plt.legend()
plt.grid(True, alpha=0.3)

# Adjust spacing and display the chart
plt.tight_layout()
plt.show()


#### Interpretation — Overall Job Satisfaction Trends (2004–2024)

The results show that average job satisfaction among employed respondents decreased from **8.21 in 2004 to 7.82 in 2024**, a decline of **0.39 points** on the 0–10 scale.

Despite this decline, the median job satisfaction score remained at 8 throughout the period.

The yearly averages fluctuated rather than decreasing continuously. The highest average was recorded in 2004 (8.21), while the lowest was recorded in 2024 (7.82).

The chart highlights three major economic events:

- **2009:** Global financial crisis.
- **2015:** Swiss franc shock following the removal of the EUR/CHF exchange-rate floor.
- **2020:** COVID-19 pandemic.

However, these markers do not establish whether the events caused changes in job satisfaction. Further comparisons around these periods are needed.

The results are descriptive and unweighted. Changes in the composition of the survey sample may also influence the yearly averages.


#### 6.1.2 Job Satisfaction Trends (2004–2024)

##### Purpose

We examine how average job satisfaction has changed between 2004 and 2024 and whether noticeable changes occurred around major economic events.

##### Analysis

We calculate:

1. **Yearly average satisfaction:** Mean job satisfaction for each survey year.
2. **Yearly observations:** Number of valid responses for each year.
3. **Overall change:** Difference in average job satisfaction between 2004 and 2024.
4. **Visualisation:** Line chart showing yearly average job satisfaction.
5. **Economic events:** Highlight the 2009 financial crisis, 2015 Swiss franc shock and 2020 COVID-19 pandemic.

##### Objective

This analysis provides an overview of long-term changes in job satisfaction and identifies periods that may require further investigation.

The economic-event markers are descriptive and do not establish that these events caused changes in job satisfaction.

In [ ]:
# 6.1.2 Additional analysis — Low Job Satisfaction Over Time

# 1. Calculate the percentage of low satisfaction (0–5) by year
yearly_low_satisfaction = (
    employed_sample
    .assign(
        LOW_SATISFACTION=lambda df: df["JOB_SATISFACTION"] <= 5
    )
    .groupby("YEAR")
    .agg(
        Observations=("JOB_SATISFACTION", "count"),
        Low_Satisfaction_Percentage=("LOW_SATISFACTION", "mean")
    )
    .reset_index()
)

yearly_low_satisfaction["Low_Satisfaction_Percentage"] *= 100

# 2. Display yearly results
print("Percentage of low job satisfaction (0–5) by year:")
display(yearly_low_satisfaction.round(2))

# 3. Plot yearly low satisfaction
plt.figure(figsize=(12, 5))

plt.plot(
    yearly_low_satisfaction["YEAR"],
    yearly_low_satisfaction["Low_Satisfaction_Percentage"],
    marker="o"
)

# Mark major economic events
events = {
    2009: "Financial crisis",
    2015: "Swiss franc shock",
    2020: "COVID-19"
}

for year, label in events.items():
    plt.axvline(
        x=year,
        color="gray",
        linestyle="--",
        alpha=0.6
    )

plt.title("Low Job Satisfaction (0–5) by Year, 2004–2024")
plt.xlabel("Survey year")
plt.ylabel("Share of observations (%)")
plt.xticks(range(2004, 2025, 2))
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 4. Compare 2004 and 2024
low_2004 = yearly_low_satisfaction.loc[
    yearly_low_satisfaction["YEAR"] == 2004,
    "Low_Satisfaction_Percentage"
].iloc[0]

low_2024 = yearly_low_satisfaction.loc[
    yearly_low_satisfaction["YEAR"] == 2024,
    "Low_Satisfaction_Percentage"
].iloc[0]

print(f"Low satisfaction in 2004: {low_2004:.2f}%")
print(f"Low satisfaction in 2024: {low_2024:.2f}%")
print(f"Change: {low_2024 - low_2004:+.2f} percentage points")

##### Additional Findings — Low Job Satisfaction (0–5)

The percentage of observations reporting low job satisfaction increased from 6.27% in 2004 to 7.56% in 2024, representing an increase of 1.29 percentage points.

- The lowest percentage was recorded in 2014 (5.26%).
- The highest percentage was recorded in 2023 (7.80%).
- Between 2004 and 2016, low satisfaction generally fluctuated between approximately 5% and 7%.
- From 2017 onwards, low satisfaction was generally higher, reaching 7.14% in 2020 and 7.71% in 2021.
- The percentage remained above 7.5% between 2021 and 2024.

These findings complement the decline in average job satisfaction from 8.21 in 2004 to 7.82 in 2024. Both indicators suggest that job satisfaction has weakened over the study period.

However, the percentage reporting low satisfaction remains relatively small, and changes may partly reflect differences in sample composition across years.

The analysis is descriptive and unweighted. It does not establish that economic events caused the observed changes.

#### 6.1.3 Job Satisfaction Trends by Panel Entry Group

##### Purpose

The Swiss Household Panel follows respondents across multiple years, but new respondents also enter the panel over time.

We examine whether the overall job satisfaction trend differs between respondents who first appear in our dataset at different times.

##### Analysis

We divide respondents into three groups according to their first observed year in the dataset:

1. Before 2014
2. 2014–2019
3. 2020–2024

For each group and survey year, we calculate:

- Number of respondent-year observations.
- Average job satisfaction.
- Percentage reporting low job satisfaction (0–5).

We then compare trends using two line charts.

##### Limitations

The first observed year in our 2004–2024 dataset is not necessarily the respondent's actual entry year into the Swiss Household Panel.

The results are descriptive and unweighted. Differences between groups may reflect age, occupation or other characteristics, rather than panel entry alone.

In [ ]:
# 6.1.3 Job Satisfaction Trends by Panel Entry Group

import pandas as pd
import matplotlib.pyplot as plt

# 1. Identify each respondent's first observed year
# Use the full dataset so entry is not defined by first employment
first_year = (
    full_dataset
    .groupby("IDPERS")["YEAR"]
    .min()
    .rename("FIRST_OBSERVED_YEAR")
)

# 2. Add the first observed year to the employed sample
entry_sample = employed_sample.copy()

entry_sample = entry_sample.join(
    first_year,
    on="IDPERS"
)

# 3. Divide respondents into entry groups
entry_sample["ENTRY_GROUP"] = pd.cut(
    entry_sample["FIRST_OBSERVED_YEAR"],
    bins=[2003, 2013, 2019, 2024],
    labels=[
        "Before 2014",
        "2014–2019",
        "2020–2024"
    ]
)

# 4. Define low satisfaction (0–5)
entry_sample["LOW_SATISFACTION"] = (
    entry_sample["JOB_SATISFACTION"] <= 5
)

# 5. Calculate yearly results by entry group
entry_trends = (
    entry_sample
    .groupby(["YEAR", "ENTRY_GROUP"], observed=True)
    .agg(
        Observations=("JOB_SATISFACTION", "count"),
        Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
        Low_Satisfaction_Percentage=("LOW_SATISFACTION", "mean")
    )
    .reset_index()
)

entry_trends["Low_Satisfaction_Percentage"] *= 100

# 6. Display yearly results
print("Job satisfaction by survey year and entry group:")
display(entry_trends.round(2))

# 7. Check the number of observations in each entry group
entry_group_counts = (
    entry_sample["ENTRY_GROUP"]
    .value_counts()
    .sort_index()
)

print("\nObservations by entry group:")
display(entry_group_counts.to_frame("Observations"))

# 8. Check whether any observations are missing an entry group
missing_entry_group = entry_sample["ENTRY_GROUP"].isna().sum()

print("\nObservations without an entry group:", missing_entry_group)

# 9. Plot average satisfaction by entry group
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for group, group_data in entry_trends.groupby(
    "ENTRY_GROUP",
    observed=True
):
    axes[0].plot(
        group_data["YEAR"],
        group_data["Mean_Satisfaction"],
        marker="o",
        label=str(group)
    )

axes[0].set_title("Average Job Satisfaction by Entry Group")
axes[0].set_xlabel("Survey year")
axes[0].set_ylabel("Mean satisfaction (0–10)")
axes[0].set_xticks(range(2004, 2025, 2))
axes[0].grid(alpha=0.3)
axes[0].legend()

# 10. Plot low satisfaction by entry group
for group, group_data in entry_trends.groupby(
    "ENTRY_GROUP",
    observed=True
):
    axes[1].plot(
        group_data["YEAR"],
        group_data["Low_Satisfaction_Percentage"],
        marker="o",
        label=str(group)
    )

axes[1].set_title("Low Job Satisfaction (0–5) by Entry Group")
axes[1].set_xlabel("Survey year")
axes[1].set_ylabel("Share of observations (%)")
axes[1].set_xticks(range(2004, 2025, 2))
axes[1].grid(alpha=0.3)
axes[1].legend()

# 11. Finalise charts
plt.tight_layout()
plt.show()

##### Results and Interpretation

The analysis compares job satisfaction across three groups based on respondents' first observed year in the 2004–2024 dataset.

**1. Sample Composition**

- Before 2014: 105,563 observations (84.61%).
- 2014–2019: 2,901 observations (2.33%).
- 2020–2024: 16,305 observations (13.07%).
- All 124,769 observations were assigned to an entry group, with no missing values.

The group first observed before 2014 accounts for most observations. The 2014–2019 group is considerably smaller, so its yearly results may fluctuate more.

**2. Average Job Satisfaction by Entry Group**

- Before 2014: Average satisfaction declines from 8.21 in 2004 to 7.81 in 2024, a decrease of 0.40 points.
- 2014–2019: Average satisfaction decreases from 7.94 in 2014 to 7.75 in 2024. However, the trend fluctuates considerably, reaching 7.55 in 2017.
- 2020–2024: Average satisfaction decreases from 8.04 in 2020 to 7.83 in 2024.

The decline in satisfaction is visible across all three entry groups over their respective observation periods.

**3. Low Job Satisfaction (0–5)**

- Before 2014: Low satisfaction increases from 6.27% in 2004 to 6.94% in 2024.
- 2014–2019: Low satisfaction fluctuates considerably, reaching 12.40% in 2017 before declining to 5.28% in 2024.
- 2020–2024: Low satisfaction increases from 7.47% in 2020 to 8.71% in 2024.

The 2014–2019 group shows greater fluctuations, partly reflecting its smaller yearly sample sizes.

**4. Implications for the Overall Trend**

The decline in average job satisfaction is not limited to newly observed respondents.

The group first observed before 2014 shows a long-term decline, even when newer entry groups are analysed separately.

However, this does not prove that the same individuals became less satisfied over time. The composition of each entry group may still change because respondents leave or return to the survey.

**5. Limitations**

- Entry groups are defined using the first observed year in our dataset, not necessarily the actual year respondents joined the Swiss Household Panel.
- The groups cover different observation periods, limiting direct comparisons.
- Respondents may appear in multiple years, and the analysis does not control for individual characteristics.
- The analysis is descriptive and unweighted.

**Conclusion**

The results suggest that the overall decline in job satisfaction cannot be explained solely by the introduction of newer respondents.

The long-term decline is also visible among respondents first observed before 2014. However, changes in the composition of this group may still contribute to the observed trend.

### 6.2 Job Satisfaction by Occupational Group

#### Purpose

We examine how job satisfaction differs between occupational groups and how these differences have changed between 2004 and 2024.

Occupations are classified using the International Standard Classification of Occupations (ISCO), with major groups coded from 0 to 9.

#### Descriptive Analysis

We calculate and compare:

1. **Mean job satisfaction:** The average satisfaction score for each occupational group in each survey year.
2. **Sample size:** The number of observations in each occupational group.
3. **Changes over time:** The difference in average satisfaction between 2004 and 2024.
4. **Occupational rankings:** How occupational groups rank according to their average job satisfaction.
5. **Satisfaction gap:** The difference between the highest and lowest occupational group averages in each year.

We use tables and line charts to examine these differences and identify patterns over time.

#### Small Sample Sizes

Some occupational groups contain very few respondents, particularly ISCO group 0 (Armed Forces).

We exclude this group from selected comparisons and visualisations because its small sample size may produce unstable averages.

The remaining nine occupational groups are used for the main comparisons.

#### Interpretation

This analysis helps us understand whether job satisfaction differs consistently across occupations and whether these differences have widened or narrowed over time.

However, the results are descriptive and unweighted. Differences between occupational groups may also reflect differences in age, sex, employment conditions and other characteristics.

The analysis does not establish whether occupation itself causes differences in job satisfaction.

In [ ]:
# Calculate mean job satisfaction for each occupation and year
occupation_yearly = (
    employed_sample
    .groupby(["YEAR", "OCCUPATION"])["JOB_SATISFACTION"]
    .agg(["count", "mean"])
    .reset_index()
)

# Rename columns for readability
occupation_yearly = occupation_yearly.rename(
    columns={
        "count": "Respondents",
        "mean": "Mean satisfaction"
    }
)

# Display the first 15 rows
print("Job satisfaction by year and occupation:")
display(occupation_yearly.head(15))

# Check the number of occupation groups in each year
print("\nOccupation groups represented each year:")
print(
    occupation_yearly.groupby("YEAR")["OCCUPATION"].nunique()
)

In [ ]:
# Define occupation names using the ISCO major groups
occupation_names = {
    1: "Managers",
    2: "Professionals",
    3: "Technicians",
    4: "Clerical support",
    5: "Service and sales",
    6: "Agricultural workers",
    7: "Craft and trades",
    8: "Machine operators",
    9: "Elementary occupations"
}

# Exclude armed forces (code 0) from this chart
occupation_plot = occupation_yearly[
    occupation_yearly["OCCUPATION"].isin(occupation_names)
].copy()

# Create the figure
plt.figure(figsize=(13, 7))

# Draw one line for each occupational group
for code, name in occupation_names.items():

    # Select observations belonging to this occupation
    group = occupation_plot[
        occupation_plot["OCCUPATION"] == code
    ]

    # Plot average satisfaction across survey years
    plt.plot(
        group["YEAR"],
        group["Mean satisfaction"],
        marker="o",
        markersize=3,
        linewidth=1.5,
        label=name
    )

# Add the chart title and axis labels
plt.title("Job Satisfaction by Occupation in Switzerland (2004–2024)")
plt.xlabel("Survey year")
plt.ylabel("Mean job satisfaction (0–10)")

# Display every survey year
plt.xticks(range(2004, 2025), rotation=45)

# Use the full satisfaction scale
plt.ylim(0, 10)

# Add the legend outside the chart
plt.legend(
    title="Occupation",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

# Add a grid
plt.grid(True, alpha=0.3)

# Adjust spacing and display the chart
plt.tight_layout()
plt.show()

In [ ]:
# Create a table showing respondent counts by occupation and year
occupation_counts = occupation_yearly.pivot(
    index="YEAR",
    columns="OCCUPATION",
    values="Respondents"
)

# Replace numeric occupation codes with readable names
occupation_counts = occupation_counts.rename(
    columns={
        0: "Armed forces",
        **occupation_names
    }
)

# Display the table
print("Respondents by occupation and survey year:")
display(occupation_counts)

# Find the smallest annual sample size for each occupation
print("\nSmallest yearly sample size by occupation:")
print(occupation_counts.min().sort_values())

# Count how many occupation-year groups have fewer than 30 respondents
small_groups = (occupation_counts < 30).sum()

print("\nNumber of years with fewer than 30 respondents:")
print(small_groups[small_groups > 0])

In [ ]:
# Select occupation results for 2004 and 2024
comparison = occupation_yearly[
    occupation_yearly["YEAR"].isin([2004, 2024])
    & occupation_yearly["OCCUPATION"].isin(occupation_names)
].copy()

# Reshape the table to show both years side by side
comparison_table = comparison.pivot(
    index="OCCUPATION",
    columns="YEAR",
    values="Mean satisfaction"
)

# Rename the occupation codes
comparison_table.index = comparison_table.index.map(
    occupation_names
)

# Calculate the change between 2004 and 2024
comparison_table["Change"] = (
    comparison_table[2024] - comparison_table[2004]
).round(2)

# Add respondent counts for 2004 and 2024
sample_sizes = comparison.pivot(
    index="OCCUPATION",
    columns="YEAR",
    values="Respondents"
)

# Replace occupation codes with names
sample_sizes.index = sample_sizes.index.map(occupation_names)

# Add sample sizes to the comparison table
comparison_table["Respondents 2004"] = sample_sizes[2004]
comparison_table["Respondents 2024"] = sample_sizes[2024]

# Sort from the largest decrease to the largest increase
comparison_table = comparison_table.sort_values("Change")

# Display the results
print("Change in mean job satisfaction by occupation (2004–2024):")
display(comparison_table)

In [ ]:
# Use unrounded averages for accurate occupational rankings
ranking_table = (
    comparison.pivot(
        index="OCCUPATION",
        columns="YEAR",
        values="Mean satisfaction"
    )
)

ranking_table.index = ranking_table.index.map(occupation_names)

# Rank occupations by satisfaction in each year
# Rank 1 means the highest average satisfaction
ranking_table["Rank 2004"] = (
    ranking_table[2004]
    .rank(ascending=False, method="min")
    .astype(int)
)

ranking_table["Rank 2024"] = (
    ranking_table[2024]
    .rank(ascending=False, method="min")
    .astype(int)
)

# Calculate the change in ranking
# A negative number means an occupation moved up
ranking_table["Rank change"] = (
    ranking_table["Rank 2024"] - ranking_table["Rank 2004"]
)

# Sort occupations by their 2004 ranking
ranking_table = ranking_table.sort_values("Rank 2004")

# Display the results
print("Occupation satisfaction rankings: 2004 vs 2024")
display(ranking_table)

In [ ]:
# Exclude armed forces because their yearly sample sizes are very small
civilian_occupation_yearly = occupation_yearly[
    occupation_yearly["OCCUPATION"] != 0
].copy()

# Find the highest and lowest occupation averages each year
occupation_gap = (
    civilian_occupation_yearly
    .groupby("YEAR")["Mean satisfaction"]
    .agg(["max", "min"])
)

# Rename the columns
occupation_gap = occupation_gap.rename(
    columns={
        "max": "Highest mean",
        "min": "Lowest mean"
    }
)

# Calculate the difference between the highest and lowest averages
occupation_gap["Gap"] = (
    occupation_gap["Highest mean"]
    - occupation_gap["Lowest mean"]
).round(2)

# Display the results
print("Job satisfaction gap between occupations (2004–2024):")
display(occupation_gap)

# Compare the first and last years
print("Gap in 2004:", occupation_gap.loc[2004, "Gap"])
print("Gap in 2024:", occupation_gap.loc[2024, "Gap"])

In [ ]:
# Create a chart showing the occupational satisfaction gap over time
plt.figure(figsize=(11, 5))

# Plot the gap for each survey year
plt.plot(
    occupation_gap.index,
    occupation_gap["Gap"],
    marker="o"
)

# Add a title and axis labels
plt.title("Job Satisfaction Gap Between Occupations (2004–2024)")
plt.xlabel("Survey year")
plt.ylabel("Gap in mean satisfaction (points)")

# Show every survey year
plt.xticks(range(2004, 2025), rotation=45)

# Start the vertical axis at zero
plt.ylim(bottom=0)

# Add a grid
plt.grid(True, alpha=0.3)

# Adjust spacing and display the chart
plt.tight_layout()
plt.show()


#### Interpretation — Job Satisfaction by Occupational Group (2004–2024)

The results show that average job satisfaction differs between occupational groups, although the differences are generally relatively small on the 0–10 scale.

**1. Changes between 2004 and 2024**

- Average job satisfaction decreased across all nine civilian occupational groups.
- The declines ranged from approximately **0.16 to 0.53 points**.
- Service and Sales workers experienced the largest decline (−0.53), followed by Professionals (−0.48).
- Machine Operators experienced the smallest decline (−0.16).

This suggests that the overall decline in job satisfaction was not limited to one occupational group.

**2. Occupational Rankings**

- Agricultural Workers had the highest average job satisfaction in both 2004 and 2024.
- Managers remained in second place in both years.
- Service and Sales fell from third place in 2004 to sixth place in 2024.
- Craft and Trades moved from sixth place to third place.

These changes show that occupational rankings were not completely stable over time.

**3. Satisfaction Gap Between Occupations**

The difference between the highest and lowest occupational averages was:

- **2004:** 0.69 points.
- **2024:** 0.62 points.

This represents a reduction of **0.07 points** between the two years.

However, the gap fluctuated across the period rather than decreasing continuously.

**4. Sample Size Considerations**

The Armed Forces category (ISCO group 0) was excluded from the comparative charts and rankings because it contained very few observations each year.

The other occupational groups also have different sample sizes, which should be considered when interpreting their averages.

**5. Overall Interpretation**

Occupation appears to be associated with differences in reported job satisfaction, but these differences are not constant over time.

Further statistical analysis would be needed to examine whether occupational differences are statistically significant and whether they vary across economic periods.

These results are descriptive and unweighted. They do not establish that occupation or economic events caused changes in job satisfaction.


#### 6.2.1 Occupational Differences in Mean, Median and Low Job Satisfaction

##### Purpose

This analysis examines how job satisfaction differs between occupational groups using three measures:

- Mean satisfaction: the average job satisfaction score.
- Median satisfaction: the middle satisfaction score.
- Low satisfaction: the percentage of observations reporting a score between 0 and 5.

Comparing these measures provides a more complete picture of occupational differences than using the mean alone.

##### Method

We group employed observations by their ISCO occupational category and calculate the number of observations, mean satisfaction, median satisfaction and percentage reporting low satisfaction.

The analysis covers 2004–2024 and uses unweighted respondent-year observations.

In [ ]:
# 6.2.1 Occupational Differences in Job Satisfaction

import pandas as pd
import matplotlib.pyplot as plt

# 1. Define readable occupation labels
occupation_labels = {
    0: "Armed Forces",
    1: "Managers",
    2: "Professionals",
    3: "Technicians",
    4: "Clerical Support",
    5: "Service and Sales",
    6: "Agricultural Workers",
    7: "Craft and Trades",
    8: "Machine Operators",
    9: "Elementary Occupations"
}

# 2. Create a separate dataset without changing employed_sample
occupation_sample = employed_sample.copy()

occupation_sample["OCCUPATION_LABEL"] = (
    occupation_sample["OCCUPATION"]
    .map(occupation_labels)
)

# 3. Identify low job satisfaction (0–5)
occupation_sample["LOW_SATISFACTION"] = (
    occupation_sample["JOB_SATISFACTION"] <= 5
)

# 4. Calculate occupational summary statistics
occupation_summary = (
    occupation_sample
    .groupby(["OCCUPATION", "OCCUPATION_LABEL"])
    .agg(
        Observations=("JOB_SATISFACTION", "count"),
        Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
        Median_Satisfaction=("JOB_SATISFACTION", "median"),
        Low_Satisfaction_Percentage=("LOW_SATISFACTION", "mean")
    )
    .reset_index()
)

occupation_summary["Low_Satisfaction_Percentage"] *= 100

# 5. Sort by mean satisfaction
occupation_summary = occupation_summary.sort_values(
    "Mean_Satisfaction",
    ascending=False
)

print("Job satisfaction by occupational group (2004–2024):")
display(occupation_summary.round(2))

# 6. Check total observations
print(
    "\nTotal observations:",
    occupation_summary["Observations"].sum()
)

# 7. Plot mean satisfaction by occupation
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].barh(
    occupation_summary["OCCUPATION_LABEL"],
    occupation_summary["Mean_Satisfaction"]
)

axes[0].invert_yaxis()
axes[0].set_title("Mean Job Satisfaction by Occupation")
axes[0].set_xlabel("Mean satisfaction (0–10)")
axes[0].set_xlim(0, 10)
axes[0].grid(axis="x", alpha=0.3)

# 8. Plot percentage reporting low satisfaction
axes[1].barh(
    occupation_summary["OCCUPATION_LABEL"],
    occupation_summary["Low_Satisfaction_Percentage"]
)

axes[1].invert_yaxis()
axes[1].set_title("Low Job Satisfaction (0–5) by Occupation")
axes[1].set_xlabel("Share of observations (%)")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

##### Results and Interpretation

**1. Average Job Satisfaction**

Average job satisfaction differs across occupational groups.

- Agricultural Workers report the highest mean satisfaction (8.34), followed by Managers (8.08).
- Professionals and Service and Sales both report an average of 8.00.
- Clerical Support has the lowest mean among the nine main occupational groups (7.78).
- Armed Forces has the lowest overall mean (7.55), but this group contains only 131 observations.

Excluding Armed Forces, the difference between the highest and lowest occupational means is 0.56 points.

**2. Median Job Satisfaction**

The median satisfaction score is 8 across all 10 occupational groups.

This suggests that the typical reported satisfaction score is similar across occupations, even though their averages differ.

**3. Low Job Satisfaction (0–5)**

The percentage reporting low satisfaction shows clearer differences between occupational groups.

- Elementary Occupations have the highest percentage of low satisfaction (11.30%).
- Machine Operators (9.40%) and Clerical Support (8.53%) also report relatively high percentages.
- Agricultural Workers have the lowest percentage (3.98%), followed by Managers (5.13%) and Professionals (5.19%).

The difference between Agricultural Workers and Elementary Occupations is 7.32 percentage points.

**4. Comparing Mean and Low Satisfaction**

Occupational groups with similar average satisfaction do not necessarily have the same percentage of dissatisfied respondents.

For example, Professionals and Service and Sales both have a mean satisfaction score of 8.00. However, low satisfaction is reported by 5.19% of Professionals compared with 7.78% of Service and Sales.

This suggests that examining the percentage of low satisfaction provides additional information beyond average scores.

**5. Limitations**

- The analysis uses unweighted respondent-year observations, so individuals may appear multiple times.
- Occupational groups have different sample sizes.
- Armed Forces has only 131 observations, making its results less reliable.
- Differences may reflect age, sex, employment conditions or other characteristics.
- The analysis is descriptive and does not establish that occupation causes differences in job satisfaction.

**Conclusion**

Job satisfaction differs across occupational groups, particularly when examining the percentage reporting low satisfaction.

Agricultural Workers and Managers generally report higher satisfaction, while Elementary Occupations, Machine Operators and Clerical Support show relatively higher levels of low satisfaction.

These findings support further analysis of occupational differences over time and during periods of economic stress.


#### 6.2.2 Job Satisfaction by Employment Arrangement

##### Purpose

This analysis examines how job satisfaction differs between employment arrangements among employed respondents in Switzerland from 2004 to 2024.

It addresses the employment-arrangement component of RQ1.1 and extends the preliminary 2024 analysis in Section 4.

##### Employment Categories

We use the `EMPLOYMENT_STATUS` variable and focus on four categories:

- **1:** Full-time employment (37+ hours per week).
- **2:** Part-time employment (5–36 hours per week).
- **3:** Part-time employment (1–4 hours per week).
- **5:** Working in a family business.

Other categories, including education, retirement and unemployment, are excluded from this comparison.

These categories describe reported main activities and do not necessarily capture every type of employment arrangement.

##### Method

Using the final employed sample, we calculate:

1. Number of observations in each employment category.
2. Mean job satisfaction.
3. Median job satisfaction.
4. Percentage reporting low job satisfaction (0–5).

We also compare average satisfaction across the four categories using a bar chart.

The analysis uses unweighted respondent-year observations.


In [ ]:
# 6.2.2 Job Satisfaction by Employment Arrangement

# Define employment arrangement categories
employment_labels = {
    1: "Full-time (37+ hours)",
    2: "Part-time (5–36 hours)",
    3: "Part-time (1–4 hours)",
    5: "Family business"
}

# Create a separate dataset without changing employed_sample
employment_arrangement_sample = employed_sample[
    employed_sample["EMPLOYMENT_STATUS"].isin(employment_labels)
].copy()

# Add readable employment labels
employment_arrangement_sample["EMPLOYMENT_LABEL"] = (
    employment_arrangement_sample["EMPLOYMENT_STATUS"]
    .map(employment_labels)
)

# Identify low job satisfaction (0–5)
employment_arrangement_sample["LOW_SATISFACTION"] = (
    employment_arrangement_sample["JOB_SATISFACTION"] <= 5
)

# Calculate summary statistics
employment_arrangement_summary = (
    employment_arrangement_sample
    .groupby(["EMPLOYMENT_STATUS", "EMPLOYMENT_LABEL"])
    .agg(
        Observations=("JOB_SATISFACTION", "count"),
        Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
        Median_Satisfaction=("JOB_SATISFACTION", "median"),
        Low_Satisfaction_Percentage=("LOW_SATISFACTION", "mean")
    )
    .reset_index()
    .sort_values("EMPLOYMENT_STATUS")
)

# Convert low satisfaction to percentages
employment_arrangement_summary["Low_Satisfaction_Percentage"] *= 100

# Display the summary
print("Job satisfaction by employment arrangement (2004–2024):")
display(employment_arrangement_summary.round(2))

# Check sample coverage
print("\nTotal employed observations:", len(employed_sample))
print("Observations in selected categories:",
      len(employment_arrangement_sample))
print("Observations excluded:",
      len(employed_sample) - len(employment_arrangement_sample))

print("\nCoverage (%):",
      round(
          len(employment_arrangement_sample) / len(employed_sample) * 100,
          2
      ))

In [ ]:
# Visualise mean job satisfaction by employment arrangement

plt.figure(figsize=(10, 5))

plt.bar(
    employment_arrangement_summary["EMPLOYMENT_LABEL"],
    employment_arrangement_summary["Mean_Satisfaction"]
)

plt.title("Mean Job Satisfaction by Employment Arrangement (2004–2024)")
plt.xlabel("Employment arrangement")
plt.ylabel("Mean job satisfaction (0–10)")
plt.ylim(0, 10)

plt.xticks(rotation=15, ha="right")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


##### Results and Interpretation

The analysis compares job satisfaction across four employment-arrangement categories from 2004 to 2024.

**1. Sample Coverage**

- The analysis includes **104,638 observations**, representing **83.87%** of the final employed sample.
- The remaining 20,131 observations belong to other employment-status categories or have missing values.
- Full-time and part-time employment (5–36 hours) account for most observations.

**2. Average Job Satisfaction**

- Family-business workers reported the highest average satisfaction (**8.47**).
- Part-time workers employed for 1–4 hours per week followed (**8.37**).
- Part-time workers employed for 5–36 hours reported an average of **7.96**.
- Full-time workers reported the lowest average (**7.91**).

The difference between the highest and lowest averages was **0.56 points** on the 0–10 scale.

**3. Low Job Satisfaction**

- Part-time workers employed for 5–36 hours had the highest percentage of low satisfaction (**6.74%**).
- Family-business workers had the lowest percentage (**4.16%**).
- The median satisfaction score was **8 across all four categories**.

**4. Interpretation and Limitations**

The results suggest that job satisfaction differs between employment arrangements, although the differences are relatively small.

The family-business and 1–4-hour part-time categories contain substantially fewer observations than the two larger employment groups. Their averages should therefore be interpreted cautiously.

The analysis covers selected employment-status categories and excludes other employed respondents whose main activity falls outside these categories.

The results are descriptive and unweighted. They do not establish that employment arrangements cause differences in job satisfaction.


### 6.3 Job Satisfaction by Sex

#### Purpose

We examine whether average job satisfaction differs between men and women in Switzerland and how these differences have changed between 2004 and 2024.

This analysis helps us identify whether the satisfaction gap between men and women has remained stable, increased or decreased over time.

#### Descriptive Analysis

For each survey year, we calculate:

1. **Mean job satisfaction:** The average satisfaction score for men and women separately.
2. **Sample size:** The number of observations for each group.
3. **Satisfaction gap:** The difference between women's and men's average job satisfaction.

We then create tables and line charts to compare satisfaction levels and changes over time.

#### Sex Categories

The SHP variable `SEX` uses the following codes:

- `1`: Men
- `2`: Women
- `3`: Other

The main comparison focuses on men and women because the employed sample contains only one observation coded as `3`.

This observation is excluded from the men-versus-women comparison due to the insufficient sample size.

#### Interpreting the Satisfaction Gap

We calculate the gap as:

**Satisfaction gap = Mean satisfaction (Women) − Mean satisfaction (Men)**

- **Positive gap:** Women report higher average job satisfaction.
- **Negative gap:** Men report higher average job satisfaction.
- **Zero gap:** Both groups have the same average satisfaction.

#### Limitations

The results are descriptive and unweighted.

Differences between men and women may also reflect differences in occupation, age, employment conditions and other characteristics.

The analysis therefore identifies patterns but does not establish whether sex itself causes differences in job satisfaction.

In [ ]:
# Group respondents by survey year and sex
sex_yearly = (
    employed_sample
    .groupby(["YEAR", "SEX"])["JOB_SATISFACTION"]
    .agg(["count", "mean"])
    .reset_index()
)

# Rename the columns
sex_yearly = sex_yearly.rename(
    columns={
        "count": "Respondents",
        "mean": "Mean satisfaction"
    }
)

# Replace numeric sex codes with labels
sex_yearly["SEX"] = sex_yearly["SEX"].map({
    1: "Men",
    2: "Women",
    3: "Other"
})

# Display the results
print("Job satisfaction by sex and survey year:")
display(sex_yearly)

# Check which groups are represented each year
print("\nSex groups represented by year:")
print(sex_yearly.groupby("YEAR")["SEX"].nunique())

In [ ]:
# Create a new chart
plt.figure(figsize=(11, 5))

# Plot average job satisfaction for men and women
for sex in ["Men", "Women"]:

    # Select observations for the current group
    group_data = sex_yearly[sex_yearly["SEX"] == sex]

    # Draw a line for this group
    plt.plot(
        group_data["YEAR"],
        group_data["Mean satisfaction"],
        marker="o",
        label=sex
    )

# Add a title and axis labels
plt.title("Job Satisfaction by Sex in Switzerland (2004–2024)")
plt.xlabel("Survey year")
plt.ylabel("Mean job satisfaction (0–10)")

# Show every survey year
plt.xticks(range(2004, 2025), rotation=45)

# Use the full satisfaction scale
plt.ylim(0, 10)

# Add a legend and grid
plt.legend()
plt.grid(True, alpha=0.3)

# Adjust spacing and display the chart
plt.tight_layout()
plt.show()

In [ ]:
# Keep only men and women for this comparison
sex_comparison = sex_yearly[
    sex_yearly["SEX"].isin(["Men", "Women"])
].copy()

# Put men and women in separate columns
sex_gap = sex_comparison.pivot(
    index="YEAR",
    columns="SEX",
    values="Mean satisfaction"
)

# Calculate the difference using unrounded averages
sex_gap["Gap (Women - Men)"] = (
    sex_gap["Women"] - sex_gap["Men"]
)

# Display the results
print("Job satisfaction gap between women and men (2004–2024):")
display(sex_gap.round(2))

# Count how many years each group had higher satisfaction
print(
    "\nYears women reported higher satisfaction:",
    (sex_gap["Gap (Women - Men)"] > 0).sum()
)

print(
    "Years men reported higher satisfaction:",
    (sex_gap["Gap (Women - Men)"] < 0).sum()
)

print(
    "Years with equal averages:",
    (sex_gap["Gap (Women - Men)"] == 0).sum()
)


#### Interpretation — Job Satisfaction by Sex (2004–2024)

The results show that women reported slightly higher average job satisfaction than men in 20 of the 21 survey years.

The largest difference occurred in 2004, when women's average satisfaction was **0.16 points higher** than men's.

The difference generally became smaller over time, although it fluctuated between years.

In 2024, men reported slightly higher average satisfaction than women, with a difference of **0.04 points**.

Overall, the differences between men and women were relatively small compared with the full 0–10 satisfaction scale.

These results are descriptive and unweighted. Further statistical analysis would be needed to determine whether the differences are statistically significant.


#### 6.3.1 Additional Satisfaction Measures by Sex

##### Purpose

This analysis extends the existing comparison of job satisfaction between men and women.

We calculate the number of observations, mean satisfaction, median satisfaction and percentage reporting low satisfaction (0–5).

These measures help identify differences that may not be visible when comparing averages alone.

The analysis uses unweighted respondent-year observations from 2004–2024.

In [ ]:
# 6.3.1 Additional Satisfaction Measures by Sex

import pandas as pd
import matplotlib.pyplot as plt

sex_sample = analysis_sample.copy()

sex_labels = {
    1: "Men",
    2: "Women",
    3: "Other"
}

sex_sample["SEX_LABEL"] = sex_sample["SEX"].map(sex_labels)

sex_sample["LOW_SATISFACTION"] = (
    sex_sample["JOB_SATISFACTION"] <= 5
)

sex_summary = (
    sex_sample
    .groupby(["SEX", "SEX_LABEL"])
    .agg(
        Observations=("JOB_SATISFACTION", "count"),
        Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
        Median_Satisfaction=("JOB_SATISFACTION", "median"),
        Low_Satisfaction_Percentage=("LOW_SATISFACTION", "mean")
    )
    .reset_index()
)

sex_summary["Low_Satisfaction_Percentage"] *= 100

print("Job satisfaction by sex (2004–2024):")
display(sex_summary.round(2))

print("\nTotal observations:", sex_summary["Observations"].sum())

# Exclude the very small 'Other' category from visualisations
sex_plot = sex_summary[
    sex_summary["SEX_LABEL"].isin(["Men", "Women"])
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(
    sex_plot["SEX_LABEL"],
    sex_plot["Mean_Satisfaction"]
)
axes[0].set_title("Mean Job Satisfaction by Sex")
axes[0].set_ylabel("Mean satisfaction (0–10)")
axes[0].set_ylim(0, 10)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(
    sex_plot["SEX_LABEL"],
    sex_plot["Low_Satisfaction_Percentage"]
)
axes[1].set_title("Low Job Satisfaction (0–5) by Sex")
axes[1].set_ylabel("Share of observations (%)")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

##### Results and Interpretation

**1. Average Job Satisfaction**

- Men report an average job satisfaction score of 7.94.
- Women report a slightly higher average of 8.00.
- The difference is 0.06 points on the 0–10 scale.

The difference in average satisfaction between men and women is relatively small.

**2. Median Job Satisfaction**

Both men and women have a median satisfaction score of 8.

This indicates that the middle reported satisfaction score is the same for both groups.

**3. Low Job Satisfaction (0–5)**

- 6.15% of observations from men report low satisfaction.
- 6.91% of observations from women report low satisfaction.
- The difference is 0.76 percentage points.

Although women report slightly higher average satisfaction, they also have a higher percentage of low-satisfaction responses.

This suggests that differences between the two groups are not fully captured by comparing averages alone.

**4. Sample Size and Limitations**

- The analysis includes 60,807 observations from men and 63,958 from women.
- One observation is classified as Other. It is included in the summary table but excluded from the charts because the sample is too small for a meaningful comparison.
- The results are descriptive and unweighted.
- Individuals may appear in multiple survey years.
- Differences may reflect occupation, age, employment conditions or other characteristics.

**Conclusion**

Average job satisfaction is similar between men and women, with women reporting a slightly higher mean.

However, women also report a somewhat higher percentage of low satisfaction. This highlights the importance of considering both average satisfaction and the distribution of responses.

### 6.4 Job Satisfaction by Age Group

#### Purpose

This analysis examines how average job satisfaction differs between age groups in Switzerland and how these differences change from 2004 to 2024.

#### Approach

We will:

1. Examine the age distribution of employed respondents.
2. Define age groups based on the available data.
3. Calculate average job satisfaction and sample sizes for each age group.
4. Compare job satisfaction across age groups and survey years.
5. Visualise the trends and identify any differences over time.

#### Considerations

Age is recorded separately in each survey year, meaning respondents may move between age groups over time.

The results will be descriptive and unweighted. Differences between age groups do not necessarily mean that age itself causes differences in job satisfaction.

In [ ]:
# Check the age distribution of employed respondents

# Display summary statistics for age
print("Age distribution of employed respondents:")
display(employed_sample["AGE"].describe())

# Count missing age values
print("\nMissing age values:")
print(employed_sample["AGE"].isna().sum())

# Check how many observations fall within broad age ranges
print("\nNumber of observations by age range:")

age_ranges = pd.cut(
    employed_sample["AGE"],
    bins=[0, 24, 34, 44, 54, 64, float("inf")],
    labels=["Under 25", "25–34", "35–44", "45–54", "55–64", "65+"]
)

display(
    age_ranges.value_counts()
    .sort_index()
    .rename_axis("Age range")
    .to_frame("Observations")
)

In [ ]:
# Create a copy of the employed sample for age-group analysis
age_sample = employed_sample.copy()

# Define the age groups
age_sample["AGE_GROUP"] = pd.cut(
    age_sample["AGE"],
    bins=[0, 24, 34, 44, 54, 64, float("inf")],
    labels=["Under 25", "25–34", "35–44", "45–54", "55–64", "65+"]
)

# Check the number of observations in each age group
print("Observations by age group:")
display(
    age_sample["AGE_GROUP"]
    .value_counts(dropna=False)
    .sort_index()
)

# Check that the age groups cover all valid ages
print("\nObservations with missing age group:")
print(age_sample["AGE_GROUP"].isna().sum())

In [ ]:
# Calculate job satisfaction statistics for each age group
age_summary = (
    age_sample
    .groupby("AGE_GROUP", observed=True)["JOB_SATISFACTION"]
    .agg(["mean", "median", "count", "std"])
    .reset_index()
)

# Rename columns to make the table easier to read
age_summary = age_summary.rename(columns={
    "mean": "Mean satisfaction",
    "median": "Median satisfaction",
    "count": "Observations",
    "std": "Standard deviation"
})

# Round the statistics for presentation
age_summary["Mean satisfaction"] = age_summary["Mean satisfaction"].round(2)
age_summary["Standard deviation"] = age_summary["Standard deviation"].round(2)

# Display the results
print("Job satisfaction by age group (2004–2024):")
display(age_summary)

In [ ]:
# Calculate job satisfaction by age group and survey year
age_yearly = (
    age_sample
    .groupby(["YEAR", "AGE_GROUP"], observed=True)["JOB_SATISFACTION"]
    .agg(["mean", "count"])
    .reset_index()
)

# Rename the columns
age_yearly = age_yearly.rename(columns={
    "mean": "Mean satisfaction",
    "count": "Observations"
})

# Display the results
print("Job satisfaction by age group and year (2004–2024):")
display(age_yearly.round(2))

In [ ]:
# Create a line chart of job satisfaction by age group over time
plt.figure(figsize=(12, 6))

# Draw one line for each age group
for age_group in age_yearly["AGE_GROUP"].cat.categories:
    group_data = age_yearly[
        age_yearly["AGE_GROUP"] == age_group
    ]

    plt.plot(
        group_data["YEAR"],
        group_data["Mean satisfaction"],
        marker="o",
        markersize=3,
        label=age_group
    )

# Add chart title and axis labels
plt.title("Average Job Satisfaction by Age Group (2004–2024)")
plt.xlabel("Survey year")
plt.ylabel("Mean job satisfaction (0–10)")

# Improve chart readability
plt.xticks(range(2004, 2025, 2))
plt.ylim(7, 9.5)
plt.grid(True, alpha=0.3)
plt.legend(title="Age group")
plt.tight_layout()

# Display the chart
plt.show()

In [ ]:
# Select the first and last survey years
age_comparison = age_yearly[
    age_yearly["YEAR"].isin([2004, 2024])
].copy()

# Put the 2004 and 2024 satisfaction averages in separate columns
age_changes = age_comparison.pivot(
    index="AGE_GROUP",
    columns="YEAR",
    values="Mean satisfaction"
)

# Calculate the change in satisfaction between 2004 and 2024
age_changes["Change (2024 - 2004)"] = (
    age_changes[2024] - age_changes[2004]
).round(2)

# Display the results
print("Changes in job satisfaction by age group (2004–2024):")
display(age_changes.round(2))


#### Interpretation — Job Satisfaction by Age Group (2004–2024)

#### Differences Between Age Groups

Average job satisfaction generally increases with age, although the pattern is not perfectly consistent.

Across the full 2004–2024 period, respondents aged 25–34 reported the lowest average job satisfaction (7.66), while respondents aged 65+ reported the highest (8.78).

Respondents aged 55–64 also reported relatively high satisfaction (8.14), compared with 7.87 for those under 25.

#### Changes Between 2004 and 2024

Average job satisfaction decreased across all six age groups.

- **Under 25:** 8.12 to 7.67 (−0.45 points). 
- **25–34:** 7.91 to 7.48 (−0.44 points).
- **35–44:** 8.15 to 7.66 (−0.49 points).
- **45–54:** 8.30 to 7.83 (−0.47 points).
- **55–64:** 8.50 to 7.91 (−0.59 points).
- **65+:** 8.80 to 8.74 (−0.06 points).

The largest decline occurred among respondents aged 55–64 (−0.59 points), while the smallest decline occurred among those aged 65+ (−0.06 points).

#### Sample Size Considerations

The 65+ age group has the smallest sample, with 6,926 observations across the full period, compared with 32,967 for ages 45–54.

This means the oldest group's yearly averages may be more sensitive to changes in the sample.

#### Overall Interpretation

Older employed respondents generally report higher job satisfaction, but all age groups experienced a decline between 2004 and 2024.

These differences do not establish that ageing directly causes higher satisfaction. They may also reflect occupation, employment conditions, generational differences and changes in sample composition.

The results are descriptive and unweighted. Further statistical analysis would be needed to assess statistical significance.


#### 6.4.1 Additional Satisfaction Measures by Age

##### Purpose

This analysis extends the existing examination of job satisfaction across age groups.

We compare mean satisfaction, median satisfaction and the percentage reporting low satisfaction (0–5).

Age groups are defined as under 25, 25–34, 35–44, 45–54, 55–64 and 65+.

The analysis uses unweighted respondent-year observations from 2004–2024.

In [ ]:
# 6.4.1 Additional Satisfaction Measures by Age

import pandas as pd
import matplotlib.pyplot as plt

age_sample = analysis_sample.copy()

age_labels = [
    "Under 25",
    "25–34",
    "35–44",
    "45–54",
    "55–64",
    "65+"
]

age_sample["AGE_GROUP"] = pd.cut(
    age_sample["AGE"],
    bins=[0, 24, 34, 44, 54, 64, float("inf")],
    labels=age_labels
)

age_sample["LOW_SATISFACTION"] = (
    age_sample["JOB_SATISFACTION"] <= 5
)

age_summary = (
    age_sample
    .groupby("AGE_GROUP", observed=True)
    .agg(
        Observations=("JOB_SATISFACTION", "count"),
        Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
        Median_Satisfaction=("JOB_SATISFACTION", "median"),
        Low_Satisfaction_Percentage=("LOW_SATISFACTION", "mean")
    )
    .reset_index()
)

age_summary["Low_Satisfaction_Percentage"] *= 100

print("Job satisfaction by age group (2004–2024):")
display(age_summary.round(2))

print("\nTotal observations:", age_summary["Observations"].sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(
    age_summary["AGE_GROUP"].astype(str),
    age_summary["Mean_Satisfaction"]
)
axes[0].set_title("Mean Job Satisfaction by Age Group")
axes[0].set_ylabel("Mean satisfaction (0–10)")
axes[0].set_ylim(0, 10)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(
    age_summary["AGE_GROUP"].astype(str),
    age_summary["Low_Satisfaction_Percentage"]
)
axes[1].set_title("Low Job Satisfaction (0–5) by Age Group")
axes[1].set_ylabel("Share of observations (%)")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

##### Results and Interpretation

**1. Average Job Satisfaction**

Average job satisfaction differs across age groups.

- Respondents aged 25–34 report the lowest average satisfaction (7.66).
- Respondents under 25 report a slightly higher average (7.87).
- Average satisfaction increases across the older age groups, reaching 8.14 among those aged 55–64.
- Respondents aged 65+ report the highest average satisfaction (8.78).

The difference between the highest and lowest age-group averages is 1.12 points on the 0–10 scale.

**2. Median Job Satisfaction**

The median satisfaction score is 8 for all age groups below 65.

Respondents aged 65+ have a higher median of 9.

This indicates that the middle reported satisfaction score is particularly high among employed respondents aged 65 and above.

**3. Low Job Satisfaction (0–5)**

The percentage reporting low satisfaction generally decreases across older age groups.

- Ages 25–34 have the highest percentage of low satisfaction (8.38%).
- Respondents under 25 have the second-highest percentage (7.61%).
- Low satisfaction decreases to 6.92% among ages 35–44 and 6.32% among ages 45–54.
- Ages 55–64 report 5.47%.
- Ages 65+ have the lowest percentage (3.02%).

The difference between the 25–34 and 65+ groups is 5.36 percentage points.

**4. Interpretation**

The results suggest that older employed respondents generally report higher job satisfaction and are less likely to report low satisfaction.

The 25–34 age group stands out as having both the lowest average satisfaction and the highest percentage of low satisfaction.

However, the relationship is not strictly increasing across every age group, as respondents under 25 report higher average satisfaction than those aged 25–34.

**5. Limitations**

- The analysis uses unweighted respondent-year observations, and individuals may appear in multiple years.
- Age groups differ in sample size and employment characteristics.
- Respondents aged 65+ are a selected group who remain employed, so their higher satisfaction may not represent all people in that age group.
- Differences may reflect occupation, employment conditions, income or other characteristics.
- The results compare different age groups and do not establish how an individual's satisfaction changes as they age.

**Conclusion**

Job satisfaction generally increases across older age groups, while the percentage reporting low satisfaction decreases.

Respondents aged 25–34 report the lowest average satisfaction and the highest percentage of low satisfaction. Those aged 65+ report the highest satisfaction and the lowest percentage of low satisfaction.

These findings suggest that age is an important characteristic to consider when examining differences in job satisfaction, although the results remain descriptive.

### 6.5 Job Satisfaction by Region

#### Purpose

This analysis examines how average job satisfaction differs across the seven major regions of Switzerland and how these differences change between 2004 and 2024.

#### Approach

We will:

1. Check the region variable and the number of observations in each region.
2. Identify the seven regions using the SHP regional classification.
3. Calculate average job satisfaction and sample sizes by region.
4. Compare regional job satisfaction across survey years.
5. Visualise regional trends and compare changes between 2004 and 2024.
6. Examine whether differences between the highest- and lowest-satisfaction regions change over time.

#### Considerations

The `REGION` variable represents the respondent's region of residence, not necessarily their region of employment.

Regional differences may reflect factors such as occupation, age, employment conditions and the composition of the survey sample.

The results will be descriptive and unweighted. Differences between regions do not establish that living in a particular region causes higher or lower job satisfaction.

In [ ]:
# Check the distribution of regions in the employed sample

# Display the number of observations for each region code
print("Observations by region (2004–2024):")
display(
    employed_sample["REGION"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Region code")
    .to_frame("Observations")
)

# Check how many region values are missing
print("\nMissing region values:")
print(employed_sample["REGION"].isna().sum())

# Check the number of unique region codes
print("\nNumber of unique regions:")
print(employed_sample["REGION"].nunique())

# Display all region codes
print("\nRegion codes present in the dataset:")
print(sorted(employed_sample["REGION"].dropna().unique()))

In [ ]:
# Create a copy of the employed sample for regional analysis
region_sample = employed_sample.copy()

# Define the SHP region labels
region_labels = {
    1: "Lake Geneva",
    2: "Espace Mittelland",
    3: "Northwestern Switzerland",
    4: "Zurich",
    5: "Eastern Switzerland",
    6: "Central Switzerland",
    7: "Ticino"
}

# Convert numerical region codes into region names
region_sample["REGION_NAME"] = region_sample["REGION"].map(region_labels)

# Check the number of observations in each region
print("Observations by region:")
display(
    region_sample["REGION_NAME"]
    .value_counts(dropna=False)
    .rename_axis("Region")
    .to_frame("Observations")
)

# Check for any region codes without a matching label
print("\nMissing region labels:")
print(region_sample["REGION_NAME"].isna().sum())

In [ ]:
# Calculate average job satisfaction by region (2004–2024)

region_summary = (
    region_sample
    .groupby("REGION_NAME")
    .agg(
        Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
        Median_Satisfaction=("JOB_SATISFACTION", "median"),
        Observations=("JOB_SATISFACTION", "count"),
        Respondents=("IDPERS", "nunique")
    )
    .reset_index()
)

# Rank regions from highest to lowest satisfaction
region_summary = region_summary.sort_values(
    "Mean_Satisfaction",
    ascending=False
)

display(region_summary.round(2))

In [ ]:
# Calculate average job satisfaction by region and year

region_yearly = (
    region_sample
    .groupby(["YEAR", "REGION_NAME"])
    .agg(
        Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
        Observations=("JOB_SATISFACTION", "count")
    )
    .reset_index()
)

# Check the number of year-region combinations
print("Number of year-region combinations:", len(region_yearly))

# Display the first 14 rows
display(region_yearly.head(14).round(2))

In [ ]:
# Plot average job satisfaction by region over time

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

# Draw one line for each region
for region in sorted(region_yearly["REGION_NAME"].unique()):

    # Select observations for this region
    region_data = region_yearly[
        region_yearly["REGION_NAME"] == region
    ]

    plt.plot(
        region_data["YEAR"],
        region_data["Mean_Satisfaction"],
        marker="o",
        markersize=3,
        linewidth=1.5,
        label=region
    )

# Add chart labels
plt.title("Average Job Satisfaction by Region (2004–2024)")
plt.xlabel("Year")
plt.ylabel("Mean Job Satisfaction (0–10)")

# Display all survey years
plt.xticks(range(2004, 2025, 2))

# Add a legend and grid
plt.legend(title="Region", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare regional job satisfaction between 2004 and 2024

# Select the first and last survey years
region_comparison = region_yearly[
    region_yearly["YEAR"].isin([2004, 2024])
]

# Display each year in a separate column
region_changes = region_comparison.pivot(
    index="REGION_NAME",
    columns="YEAR",
    values="Mean_Satisfaction"
)

# Calculate the change between 2004 and 2024
region_changes["Change"] = (
    region_changes[2024] - region_changes[2004]
)

# Rename columns
region_changes = region_changes.rename(
    columns={
        2004: "Mean_2004",
        2024: "Mean_2024"
    }
)

# Sort by largest decline
region_changes = region_changes.sort_values("Change")

display(region_changes.round(2))


#### Interpretation — Job Satisfaction by Region (2004–2024)

#### Differences Between Regions

Average job satisfaction varies across Switzerland's seven regions.

In 2004, Central Switzerland had the highest average job satisfaction (8.36), followed by Eastern Switzerland (8.30). Zurich had the lowest average (8.12).

In 2024, Eastern Switzerland had the highest average (7.99), followed by Central Switzerland (7.97). Lake Geneva had the lowest average (7.71).

The difference between the highest and lowest regional averages was 0.24 points in 2004 and 0.28 points in 2024.

#### Changes Between 2004 and 2024

Average job satisfaction decreased in all seven regions.

| Region | 2004 | 2024 | Change |
|---|---:|---:|---:|
| Ticino | 8.27 | 7.74 | -0.53 |
| Northwestern Switzerland | 8.25 | 7.72 | -0.53 |
| Lake Geneva | 8.13 | 7.71 | -0.42 |
| Central Switzerland | 8.36 | 7.97 | -0.39 |
| Espace Mittelland | 8.21 | 7.84 | -0.36 |
| Zurich | 8.12 | 7.76 | -0.36 |
| Eastern Switzerland | 8.30 | 7.99 | -0.31 |

Ticino and Northwestern Switzerland experienced the largest declines (-0.53 points each).

Eastern Switzerland experienced the smallest decline (-0.31 points).

#### Overall Interpretation

Job satisfaction declined across all seven regions between 2004 and 2024, although the size of the decline differed.

The regional gap increased slightly from 0.24 to 0.28 points, suggesting that differences between regional averages remained relatively small.

However, these results do not establish that living or working in a particular region causes higher or lower job satisfaction. Differences may also reflect occupation, age, employment conditions and changes in sample composition.

The results are descriptive and unweighted. Further statistical analysis would be needed to assess whether the regional differences are statistically significant.


#### 6.5.1 Additional Satisfaction Measures by Region

##### Purpose

This analysis extends the existing comparison of job satisfaction across Switzerland's seven major regions.

We calculate:

- Number of observations in each region.
- Mean job satisfaction.
- Median job satisfaction.
- Percentage reporting low job satisfaction (0–5).

Comparing these measures helps identify regional differences that may not be visible from average satisfaction alone.

##### Method

We use the existing regional classification in the Swiss Household Panel and calculate summary statistics for employed respondents between 2004 and 2024.

The analysis uses unweighted respondent-year observations. Individuals may appear in multiple survey years.

In [ ]:
# 6.5.1 Additional Satisfaction Measures by Region

import pandas as pd
import matplotlib.pyplot as plt

# 1. Define readable region labels
region_labels = {
    1: "Lake Geneva",
    2: "Espace Mittelland",
    3: "Northwestern Switzerland",
    4: "Zurich",
    5: "Eastern Switzerland",
    6: "Central Switzerland",
    7: "Ticino"
}

# 2. Create a separate dataset
region_sample = analysis_sample.copy()

region_sample["REGION_LABEL"] = (
    region_sample["REGION"].map(region_labels)
)

# 3. Define low job satisfaction (0–5)
region_sample["LOW_SATISFACTION"] = (
    region_sample["JOB_SATISFACTION"] <= 5
)

# 4. Calculate regional summary statistics
region_summary = (
    region_sample
    .groupby(["REGION", "REGION_LABEL"])
    .agg(
        Observations=("JOB_SATISFACTION", "count"),
        Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
        Median_Satisfaction=("JOB_SATISFACTION", "median"),
        Low_Satisfaction_Percentage=("LOW_SATISFACTION", "mean")
    )
    .reset_index()
)

region_summary["Low_Satisfaction_Percentage"] *= 100

# 5. Sort by mean satisfaction
region_summary = region_summary.sort_values(
    "Mean_Satisfaction",
    ascending=False
)

print("Job satisfaction by region (2004–2024):")
display(region_summary.round(2))

# 6. Check total observations
print(
    "\nTotal observations:",
    region_summary["Observations"].sum()
)

# 7. Plot mean satisfaction and low satisfaction
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].barh(
    region_summary["REGION_LABEL"],
    region_summary["Mean_Satisfaction"]
)

axes[0].invert_yaxis()
axes[0].set_title("Mean Job Satisfaction by Region")
axes[0].set_xlabel("Mean satisfaction (0–10)")
axes[0].set_xlim(0, 10)
axes[0].grid(axis="x", alpha=0.3)

axes[1].barh(
    region_summary["REGION_LABEL"],
    region_summary["Low_Satisfaction_Percentage"]
)

axes[1].invert_yaxis()
axes[1].set_title("Low Job Satisfaction (0–5) by Region")
axes[1].set_xlabel("Share of observations (%)")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()


##### Results and Interpretation

**1. Average Job Satisfaction**

Average job satisfaction differs across Switzerland's seven major regions.

- Eastern Switzerland reports the highest average satisfaction (8.13), followed by Central Switzerland (8.10).
- Northwestern Switzerland (7.99) and Espace Mittelland (7.97) have similar averages.
- Zurich (7.92) and Ticino (7.90) report slightly lower averages.
- Lake Geneva has the lowest average satisfaction (7.80).

The difference between the highest and lowest regional averages is 0.33 points on the 0–10 scale.

**2. Median Job Satisfaction**

All seven regions have a median satisfaction score of 8.

This suggests that the middle reported satisfaction score is similar across regions, despite differences in their averages.

**3. Low Job Satisfaction (0–5)**

The percentage reporting low satisfaction varies across regions.

- Ticino has the highest percentage of low satisfaction (8.31%), followed closely by Lake Geneva (8.26%).
- Zurich (6.68%) and Espace Mittelland (6.53%) report intermediate percentages.
- Northwestern Switzerland reports 6.27%.
- Eastern Switzerland reports 5.38%.
- Central Switzerland has the lowest percentage (4.70%).

The difference between Ticino and Central Switzerland is 3.61 percentage points.

**4. Comparing Mean and Low Satisfaction**

Regions with similar average satisfaction do not necessarily have the same percentage of low-satisfaction responses.

For example, Ticino and Zurich have relatively similar mean satisfaction scores (7.90 and 7.92), but Ticino has a higher percentage of low satisfaction (8.31% compared with 6.68%).

This demonstrates why examining low satisfaction provides additional information beyond regional averages.

**5. Limitations**

- The analysis uses unweighted respondent-year observations, and individuals may appear in multiple survey years.
- Regional sample sizes differ, with Espace Mittelland having the most observations (31,604) and Ticino the fewest (4,186).
- Regional differences may reflect occupation, age, employment conditions, industry composition or other characteristics.
- The analysis is descriptive and does not establish that geographical location causes differences in job satisfaction.

**Conclusion**

Regional differences in job satisfaction are present, although the differences in average scores are relatively small.

Eastern and Central Switzerland generally report higher average satisfaction, while Lake Geneva and Ticino show relatively higher percentages of low satisfaction.

These results suggest that region is another relevant characteristic when examining job satisfaction in Switzerland.


### 6.6 Job Satisfaction During Economic Events

**Purpose:** Examine how average job satisfaction changed around three major economic events in Switzerland: the 2009 financial crisis, the 2015 Swiss franc shock, and the 2020 COVID-19 pandemic.

**Approach:**
- Compare average job satisfaction before, during, and after each event.
- Examine whether changes differ between occupational groups.
- Compare event periods with surrounding years to identify possible changes in trends.

**Considerations:** This is an exploratory analysis using unweighted annual averages. Changes cannot be attributed directly to economic events, as other factors and changes in the survey sample may influence the results.

In [ ]:
# Define the three economic events and their comparison years

event_windows = {
    "Financial Crisis": [2008, 2009, 2010],
    "Swiss Franc Shock": [2014, 2015, 2016],
    "COVID-19 Pandemic": [2019, 2020, 2021]
}

# Display the years selected for each event
for event, years in event_windows.items():
    print(f"{event}:")
    print(f"  Before: {years[0]}")
    print(f"  Event:  {years[1]}")
    print(f"  After:  {years[2]}")
    print()

In [ ]:
# Calculate average job satisfaction for each economic event window

# Create an empty list to store the results
event_results = []

# Go through each economic event
for event, years in event_windows.items():

    # Select observations from the three comparison years
    event_data = employed_sample[
        employed_sample["YEAR"].isin(years)
    ]

    # Calculate average satisfaction and sample size by year
    yearly_summary = (
        event_data
        .groupby("YEAR")
        .agg(
            Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
            Observations=("JOB_SATISFACTION", "count")
        )
        .reset_index()
    )

    # Add the event name and period
    yearly_summary["Event"] = event

    yearly_summary["Period"] = yearly_summary["YEAR"].map({
        years[0]: "Before",
        years[1]: "During",
        years[2]: "After"
    })

    event_results.append(yearly_summary)

# Combine the results into one table
event_summary = pd.concat(event_results, ignore_index=True)

display(event_summary[[
    "Event", "Period", "YEAR", "Mean_Satisfaction", "Observations"
]].round(2))

In [ ]:

# Compare changes in job satisfaction around economic events

# Put Before, During and After into separate columns
event_changes = event_summary.pivot(
    index="Event",
    columns="Period",
    values="Mean_Satisfaction"
)

# Calculate changes between periods using unrounded averages
event_changes["Before_to_During"] = (
    event_changes["During"] - event_changes["Before"]
)

event_changes["During_to_After"] = (
    event_changes["After"] - event_changes["During"]
)

event_changes["Before_to_After"] = (
    event_changes["After"] - event_changes["Before"]
)

# Arrange columns in chronological order
event_changes = event_changes[[
    "Before",
    "During",
    "After",
    "Before_to_During",
    "During_to_After",
    "Before_to_After"
]]

# Display results rounded to 2 decimal places
display(event_changes.round(2))


In [ ]:
# Calculate job satisfaction by occupation around economic events

# Create an empty list to store the results
occupation_event_results = []

# Go through each economic event
for event, years in event_windows.items():

    # Select observations from the three comparison years
    event_data = employed_sample[
        employed_sample["YEAR"].isin(years)
    ]

    # Exclude occupation group 0 (Armed Forces) due to its small sample
    event_data = event_data[
        event_data["OCCUPATION"] != 0
    ]

    # Calculate average satisfaction and sample size by occupation and year
    occupation_summary = (
        event_data
        .groupby(["YEAR", "OCCUPATION"])
        .agg(
            Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
            Observations=("JOB_SATISFACTION", "count")
        )
        .reset_index()
    )

    # Add the event name
    occupation_summary["Event"] = event

    # Identify the period before, during or after the event
    occupation_summary["Period"] = occupation_summary["YEAR"].map({
        years[0]: "Before",
        years[1]: "During",
        years[2]: "After"
    })

    # Store the results
    occupation_event_results.append(occupation_summary)

# Combine results from all three events
occupation_event_summary = pd.concat(
    occupation_event_results,
    ignore_index=True
)

# Check the resulting dataset
print(
    "Number of occupation-event-year combinations:",
    len(occupation_event_summary)
)

display(occupation_event_summary.head(15).round(2))

In [ ]:
# Compare job satisfaction changes by occupation around each event

# Put Before, During and After into separate columns
occupation_event_changes = occupation_event_summary.pivot(
    index=["Event", "OCCUPATION"],
    columns="Period",
    values="Mean_Satisfaction"
)

# Calculate the change from before to during each event
occupation_event_changes["Before_to_During"] = (
    occupation_event_changes["During"]
    - occupation_event_changes["Before"]
)

# Calculate the change from during to after each event
occupation_event_changes["During_to_After"] = (
    occupation_event_changes["After"]
    - occupation_event_changes["During"]
)

# Calculate the overall change across the three-year window
occupation_event_changes["Before_to_After"] = (
    occupation_event_changes["After"]
    - occupation_event_changes["Before"]
)

# Arrange the columns in chronological order
occupation_event_changes = occupation_event_changes[
    [
        "Before",
        "During",
        "After",
        "Before_to_During",
        "During_to_After",
        "Before_to_After"
    ]
]

# Display the comparison table
display(occupation_event_changes.round(2))

In [ ]:
# Check sample sizes for each occupation during the economic events

# Find the smallest annual sample size for each occupation and event
occupation_sample_check = (
    occupation_event_summary
    .groupby(["Event", "OCCUPATION"])
    .agg(
        Min_Observations=("Observations", "min"),
        Max_Observations=("Observations", "max")
    )
    .reset_index()
)

# Sort to show the smallest samples first
occupation_sample_check = occupation_sample_check.sort_values(
    "Min_Observations"
)

display(occupation_sample_check)

# Check whether any occupation has fewer than 30 observations
small_samples = occupation_sample_check[
    occupation_sample_check["Min_Observations"] < 30
]

print("\nOccupation-event groups with fewer than 30 observations:")
display(small_samples)

In [ ]:
# Visualise changes in job satisfaction during economic events
import matplotlib.pyplot as plt

# Select changes from before to during each event
event_plot_data = (
    occupation_event_changes["Before_to_During"]
    .unstack("Event")
)

# Display occupational codes as whole numbers
event_plot_data.index = event_plot_data.index.astype(int)

# Arrange the events chronologically
event_plot_data = event_plot_data[
    ["Financial Crisis", "Swiss Franc Shock", "COVID-19 Pandemic"]
]

# Create a grouped bar chart
ax = event_plot_data.plot(
    kind="bar",
    figsize=(12, 6),
    width=0.8
)

# Add a horizontal reference line at zero
ax.axhline(y=0, color="black", linewidth=1)

# Add labels and title
ax.set_title("Change in Job Satisfaction by Occupation During Economic Events")
ax.set_xlabel("Occupational Group (ISCO)")
ax.set_ylabel("Change in Mean Job Satisfaction (0–10 Scale)")

# Improve readability
plt.xticks(rotation=0)
plt.legend(title="Economic Event")
plt.tight_layout()
plt.show()


#### Interpretation — Job Satisfaction During Economic Events (2008–2021)

#### 1. Overall Changes Around Economic Events

We compare average job satisfaction before, during and after three major economic events.

| Economic Event | Before | During | After | Before to During | Before to After |
|---|---:|---:|---:|---:|---:|
| Financial Crisis (2008–2010) | 8.06 | 8.02 | 8.00 | -0.03 | -0.05 |
| Swiss Franc Shock (2014–2016) | 8.05 | 7.98 | 7.96 | -0.07 | -0.09 |
| COVID-19 Pandemic (2019–2021) | 7.88 | 7.96 | 7.88 | +0.08 | 0.00 |

**Financial Crisis (2009):** Average job satisfaction decreased from 8.06 in 2008 to 8.02 in 2009, before falling further to 8.00 in 2010. The total decline was approximately 0.05 points.

**Swiss Franc Shock (2015):** Satisfaction decreased from 8.05 in 2014 to 7.98 in 2015 and 7.96 in 2016. This was the largest overall decline across the three event windows (-0.09 points).

**COVID-19 Pandemic (2020):** Satisfaction increased from 7.88 in 2019 to 7.96 in 2020 (+0.08 points), before returning to 7.88 in 2021. Unlike the other two events, there was no overall change between the first and last years.

#### 2. Differences Between Occupational Groups

Changes in job satisfaction differed across the nine occupational groups.

**Financial Crisis (2008–2010):**

- Technicians and associate professionals (ISCO 3) experienced the largest decline during the crisis (−0.14), followed by agricultural workers (ISCO 6, −0.12).
- Clerical support workers (ISCO 4) experienced an increase of +0.05 during the crisis.
- Between 2008 and 2010, agricultural workers experienced the largest overall decline (-0.22), while elementary occupations (ISCO 9) experienced the largest increase (+0.14).

**Swiss Franc Shock (2014–2016):**

- Agricultural workers (ISCO 6) experienced the largest decline during the shock (-0.29).
- Craft and related trades workers (ISCO 7) were the only group showing a small increase during the shock (+0.01).
- Between 2014 and 2016, clerical support workers (ISCO 4) experienced the largest overall decline (-0.18).

**COVID-19 Pandemic (2019–2021):**

- Job satisfaction increased during 2020 across all nine occupational groups.
- Plant and machine operators (ISCO 8) experienced the largest increase (+0.24), followed by craft and related trades workers (ISCO 7, +0.18).
- Agricultural workers (ISCO 6) also experienced an increase (+0.14).
- Between 2019 and 2021, elementary occupations (ISCO 9) experienced the largest overall increase (+0.22), while professionals and service and sales workers both experienced decreases of approximately 0.04 points.

#### 3. Sample Size Considerations

The occupation-event analysis contains 81 occupation-year combinations across three economic events, three years per event and nine occupational groups.

The smallest occupational sample contained 108 observations (ISCO 8, 2008). No occupation-event-year group contained fewer than 30 observations.

However, some occupations have substantially smaller samples than others, which may affect the stability of their averages.

#### 4. Overall Interpretation

The results suggest that changes in job satisfaction around economic events differ across occupational groups.

The financial crisis and Swiss franc shock were associated with small overall declines in job satisfaction, while satisfaction increased during the first year of COVID-19 before returning to its earlier level.

The differences between occupations suggest that economic conditions may be associated with job satisfaction differently across types of work.

However, these are descriptive comparisons based on unweighted annual averages. The results do not establish that the economic events directly caused the observed changes. Other factors, existing trends and changes in survey composition may also explain the differences.

Further statistical analysis is needed to assess whether occupational differences and changes across economic periods are statistically significant.


### 6.7 Occupational Differences Across Economic Periods

#### Purpose

This analysis examines how job satisfaction differs between occupational groups across different economic periods.

It builds on Section 6.6 by comparing occupational differences over broader periods rather than focusing only on individual event years.

This analysis contributes to Research Question 1.3, which examines how occupational differences in job satisfaction vary depending on the economic context.

#### Method

We divide the 2004–2024 study period into four groups:

1. **2004–2008:** Before and at the beginning of the global financial crisis.
2. **2009–2014:** Financial crisis aftermath and recovery.
3. **2015–2019:** Swiss franc shock and subsequent years.
4. **2020–2024:** COVID-19 and subsequent years.

For each occupational group and economic period, we calculate:

- Number of observations.
- Mean job satisfaction.
- Percentage reporting low job satisfaction (0–5).

We use two heatmaps to compare average satisfaction and low-satisfaction percentages across occupations and periods.

#### Limitations

These periods are descriptive groupings and do not represent exclusively periods of economic stress.

Differences between periods may reflect long-term trends, changes in sample composition and other factors.

The analysis is descriptive and unweighted. It does not establish that economic events caused changes in job satisfaction.

In [ ]:
# 6.7 Occupational Differences Across Economic Periods

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Create a separate dataset
period_sample = employed_sample.copy()

# 2. Define economic periods
period_labels = [
    "2004–2008",
    "2009–2014",
    "2015–2019",
    "2020–2024"
]

period_sample["ECONOMIC_PERIOD"] = pd.cut(
    period_sample["YEAR"],
    bins=[2003, 2008, 2014, 2019, 2024],
    labels=period_labels
)

# 3. Define occupation labels
occupation_labels = {
    0: "Armed Forces",
    1: "Managers",
    2: "Professionals",
    3: "Technicians",
    4: "Clerical Support",
    5: "Service and Sales",
    6: "Agricultural Workers",
    7: "Craft and Trades",
    8: "Machine Operators",
    9: "Elementary Occupations"
}

period_sample["OCCUPATION_LABEL"] = (
    period_sample["OCCUPATION"].map(occupation_labels)
)

# 4. Define low job satisfaction (0–5)
period_sample["LOW_SATISFACTION"] = (
    period_sample["JOB_SATISFACTION"] <= 5
)

# 5. Calculate satisfaction statistics by occupation and period
period_summary = (
    period_sample
    .groupby(
        ["OCCUPATION", "OCCUPATION_LABEL", "ECONOMIC_PERIOD"],
        observed=True
    )
    .agg(
        Observations=("JOB_SATISFACTION", "count"),
        Mean_Satisfaction=("JOB_SATISFACTION", "mean"),
        Low_Satisfaction_Percentage=("LOW_SATISFACTION", "mean")
    )
    .reset_index()
)

period_summary["Low_Satisfaction_Percentage"] *= 100

print("Job satisfaction by occupation and economic period:")
display(period_summary.round(2))

print(
    "\nTotal observations:",
    period_summary["Observations"].sum()
)

# 6. Exclude Armed Forces from heatmaps due to small sample size
period_plot = period_summary[
    period_summary["OCCUPATION"] != 0
].copy()

occupation_order = [
    "Managers",
    "Professionals",
    "Technicians",
    "Clerical Support",
    "Service and Sales",
    "Agricultural Workers",
    "Craft and Trades",
    "Machine Operators",
    "Elementary Occupations"
]

# 7. Prepare mean satisfaction heatmap
mean_heatmap = period_plot.pivot(
    index="OCCUPATION_LABEL",
    columns="ECONOMIC_PERIOD",
    values="Mean_Satisfaction"
)

mean_heatmap = mean_heatmap.reindex(
    index=occupation_order,
    columns=period_labels
)

# 8. Prepare low satisfaction heatmap
low_heatmap = period_plot.pivot(
    index="OCCUPATION_LABEL",
    columns="ECONOMIC_PERIOD",
    values="Low_Satisfaction_Percentage"
)

low_heatmap = low_heatmap.reindex(
    index=occupation_order,
    columns=period_labels
)

# 9. Plot both heatmaps
fig, axes = plt.subplots(1, 2, figsize=(17, 7))

sns.heatmap(
    mean_heatmap,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    linewidths=0.5,
    ax=axes[0],
    cbar_kws={"label": "Mean satisfaction"}
)

axes[0].set_title("Mean Job Satisfaction by Occupation and Period")
axes[0].set_xlabel("Economic period")
axes[0].set_ylabel("Occupational group")

sns.heatmap(
    low_heatmap,
    annot=True,
    fmt=".1f",
    cmap="OrRd",
    linewidths=0.5,
    ax=axes[1],
    cbar_kws={"label": "Low satisfaction (%)"}
)

axes[1].set_title("Low Job Satisfaction (0–5) by Occupation and Period")
axes[1].set_xlabel("Economic period")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

# 10. Compare mean satisfaction between first and last periods
period_change = mean_heatmap[
    ["2004–2008", "2020–2024"]
].copy()

period_change["Change"] = (
    period_change["2020–2024"]
    - period_change["2004–2008"]
)

period_change = period_change.sort_values("Change")

print("\nChange in mean satisfaction between first and last periods:")
display(period_change.round(2))


#### Results and Interpretation

**1. Average Job Satisfaction Across Economic Periods**

Average job satisfaction differs across occupational groups and economic periods.

- Agricultural Workers report the highest mean satisfaction in all four periods, decreasing from 8.45 in 2004–2008 to 8.26 in 2020–2024.
- Managers maintain relatively high satisfaction, declining from 8.18 to 8.05.
- Professionals decline from 8.19 to 7.88.
- Service and Sales decline from 8.16 to 7.86.
- Technicians decline from 8.16 to 7.87.
- Clerical Support reports the lowest mean satisfaction in the final period (7.68).

All nine main occupational groups have lower average satisfaction in 2020–2024 than in 2004–2008.

**2. Changes Between the First and Last Periods**

The size of the decline varies between occupational groups.

The largest decreases are observed among:

- Professionals: −0.32 points.
- Service and Sales: −0.30 points.
- Technicians: −0.29 points.
- Clerical Support: −0.23 points.

Smaller decreases are observed among:

- Managers: −0.13 points.
- Elementary Occupations: −0.08 points.
- Craft Workers: −0.05 points.
- Machine Operators: −0.05 points.

These results indicate that occupational groups have not experienced identical changes in average job satisfaction.

**3. Low Job Satisfaction (0–5)**

The percentage reporting low satisfaction also differs across occupations and periods.

- Elementary Occupations consistently report high levels of low satisfaction, increasing from 11.04% in 2004–2008 to 12.36% in 2020–2024.
- Machine Operators report 10.99% in 2004–2008, decreasing to 5.67% in 2009–2014 before increasing to 11.42% in 2020–2024.
- Clerical Support increases from 8.28% to 9.83%.
- Service and Sales increases from 7.00% to 9.54%.
- Agricultural Workers maintain relatively low percentages, increasing from 3.90% to 4.70%.
- Managers increase from 4.60% to 5.80%.

The highest percentages of low satisfaction in 2020–2024 are found among Elementary Occupations (12.36%) and Machine Operators (11.42%).

**4. Occupational Differences Across Periods**

The results show that occupational differences are not completely stable across economic periods.

Agricultural Workers maintain the highest average satisfaction throughout the study period, while the relative positions of other occupational groups change.

For example, Professionals report slightly higher average satisfaction than Managers in 2004–2008 (8.19 compared with 8.18). By 2020–2024, Managers report higher satisfaction than Professionals (8.05 compared with 7.88).

The gap between the highest and lowest mean satisfaction among the nine main occupational groups is:

- 0.63 points in 2004–2008.
- 0.55 points in 2009–2014.
- 0.57 points in 2015–2019.
- 0.58 points in 2020–2024.

This suggests that the overall occupational satisfaction gap fluctuates rather than consistently widening or narrowing.

**5. Implications for Research Question 1.3**

The findings provide descriptive evidence that occupational differences in job satisfaction vary across economic periods.

Some occupational groups, particularly Professionals, Service and Sales, and Technicians, experience larger declines between the first and last periods than others.

However, the analysis does not establish whether these differences are statistically significant.

A two-way ANOVA with an occupation-by-period interaction could be used to examine whether the relationship between occupation and job satisfaction differs significantly across periods.

A formal interaction test has not yet been conducted. Because the SHP follows respondents across multiple years, a standard two-way ANOVA would also require caution due to repeated observations from the same individuals. The current findings should therefore be interpreted as descriptive rather than statistically confirmed differences.

**6. Limitations**

- The economic periods are broad descriptive categories rather than direct measures of economic stress.
- The analysis uses unweighted respondent-year observations, and individuals may appear in multiple years.
- Changes may reflect differences in age, sex, employment conditions and sample composition.
- Armed Forces is excluded from the heatmaps because of its small sample size, although its observations remain in the summary table.
- The results do not establish that economic events caused changes in job satisfaction.

**Conclusion**

Average job satisfaction declines between the first and last periods across all nine main occupational groups, although the size of the decline differs.

Agricultural Workers maintain the highest average satisfaction, while Professionals, Service and Sales, and Technicians experience the largest declines.

These results suggest that occupational differences in job satisfaction vary across economic periods, providing descriptive evidence relevant to Research Question 1.3.



### 6.8 Statistical Testing

This section examines whether the differences observed in the descriptive analysis are statistically significant.

We use ordinary least squares (OLS) regression with standard errors clustered by respondent ID (`IDPERS`) because individuals may appear in multiple survey years.

The results describe statistical associations rather than causal effects. Survey-weighted regression and more advanced panel-data models are not included at this stage.

#### 6.8.1 RQ1.1 — Occupational Differences in Job Satisfaction

We first test whether job satisfaction differs significantly between occupational groups, controlling for survey year.

The model includes occupation as a categorical variable and survey-year fixed effects. Standard errors are clustered by respondent.


In [ ]:
# RQ1.1: Occupational differences in job satisfaction
# Civilian occupations only, with clerical workers as reference

import statsmodels.formula.api as smf

rq11_data = analysis_sample[
    ["IDPERS", "YEAR", "OCCUPATION", "JOB_SATISFACTION"]
].dropna().copy()

# Keep civilian occupational categories 1–9
rq11_data["OCCUPATION"] = pd.to_numeric(
    rq11_data["OCCUPATION"],
    errors="coerce"
)

rq11_data = rq11_data[
    rq11_data["OCCUPATION"].isin(range(1, 10))
].copy()

rq11_data["OCCUPATION"] = rq11_data["OCCUPATION"].astype(int)

# OLS with respondent-clustered standard errors
model_rq11 = smf.ols(
    formula=(
        "JOB_SATISFACTION ~ "
        "C(OCCUPATION, Treatment(reference=4)) + C(YEAR)"
    ),
    data=rq11_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": rq11_data["IDPERS"]}
)

print(model_rq11.summary())

# Joint test: Are all occupation coefficients equal to zero?
occupation_terms = [
    name for name in model_rq11.params.index
    if name.startswith("C(OCCUPATION")
]

joint_test = model_rq11.wald_test(
    " = 0, ".join(occupation_terms) + " = 0",
    scalar=True
)

print("\nJoint test of occupational differences:")
print(joint_test)

print("\nObservations:", len(rq11_data))
print("Unique respondents:", rq11_data["IDPERS"].nunique())


#### Interpretation — Occupational Differences

The regression includes 124,635 observations from 22,242 respondents across nine civilian occupational groups (2004–2024). Survey year is controlled for, and standard errors are clustered by respondent to account for repeated observations.

The joint Wald test shows statistically significant differences between occupational groups (χ²(8) = 168.22, p < 0.001).

Using clerical support workers as the reference group, agricultural workers have the largest positive difference in job satisfaction (+0.560 points), followed by managers (+0.317) and professionals (+0.231). Six of the eight occupational groups have significantly higher satisfaction than clerical support workers.

Differences for machine operators (+0.052, p = 0.404) and elementary occupations (+0.091, p = 0.126) are not statistically significant.

The model explains approximately 1% of the variation in job satisfaction (R² = 0.010). The findings show associations between occupation and satisfaction, not causal effects. Survey weights are not applied in this regression.



#### 6.8.2 RQ1.1 — Demographic and Employment Differences

This regression examines whether job satisfaction differs by age, sex, region of residence and employment arrangement, while controlling for occupation and survey year.

Standard errors are clustered by respondent. Results are interpreted as associations rather than causal effects.


In [ ]:
# RQ1.1: Demographic and employment differences

rq11_full = analysis_sample[
    [
        "IDPERS", "YEAR", "JOB_SATISFACTION",
        "OCCUPATION", "AGE", "SEX",
        "REGION", "EMPLOYMENT_STATUS"
    ]
].copy()

# Keep civilian occupations and employment categories 1, 2, 3, 5
rq11_full = rq11_full[
    rq11_full["OCCUPATION"].isin(range(1, 10))
    & rq11_full["EMPLOYMENT_STATUS"].isin([1, 2, 3, 5])
    & rq11_full["SEX"].isin([1, 2])
].dropna().copy()

# Create age groups
rq11_full["AGE_GROUP"] = pd.cut(
    rq11_full["AGE"],
    bins=[0, 24, 34, 44, 54, 64, np.inf],
    labels=["Under 25", "25–34", "35–44", "45–54", "55–64", "65+"],
    include_lowest=True
)

# Regression with clustered standard errors
model_rq11_full = smf.ols(
    formula=(
        "JOB_SATISFACTION ~ "
        "C(OCCUPATION) + C(AGE_GROUP) + C(SEX) + "
        "C(REGION) + C(EMPLOYMENT_STATUS) + C(YEAR)"
    ),
    data=rq11_full
).fit(
    cov_type="cluster",
    cov_kwds={"groups": rq11_full["IDPERS"]}
)

print(model_rq11_full.summary())

print("\nObservations:", len(rq11_full))
print("Unique respondents:", rq11_full["IDPERS"].nunique())

In [ ]:
# Joint significance tests for RQ1.1 variables

for variable in [
    "OCCUPATION",
    "AGE_GROUP",
    "SEX",
    "REGION",
    "EMPLOYMENT_STATUS"
]:
    terms = [
        name for name in model_rq11_full.params.index
        if name.startswith(f"C({variable})[")
    ]

    if not terms:
        continue

    hypothesis = " = 0, ".join(terms) + " = 0"

    test = model_rq11_full.wald_test(
        hypothesis,
        scalar=True
    )

    print(
        f"{variable}: "
        f"Chi-square = {float(test.statistic):.3f}, "
        f"p-value = {float(test.pvalue):.6g}"
    )


#### Interpretation — Demographic and Employment Differences

The regression includes 104,526 observations from 18,363 respondents across 2004–2024. It examines associations between job satisfaction and occupation, age, sex, region of residence and employment arrangement, controlling for survey year. Standard errors are clustered by respondent.

Joint Wald tests show that all five variables are statistically significant:

- Occupation: χ²(8) = 116.02, p < 0.001.
- Age group: χ²(5) = 670.26, p < 0.001.
- Sex: χ²(1) = 46.45, p < 0.001.
- Region of residence: χ²(6) = 119.89, p < 0.001.
- Employment arrangement: χ²(3) = 27.49, p < 0.001.

Compared with managers, clerical support workers report lower job satisfaction (−0.321 points, p < 0.001). Respondents aged 65+ report higher satisfaction than those under 25 (+1.088 points, p < 0.001). Sex category 2 reports higher satisfaction than category 1 (+0.156 points, p < 0.001).

Compared with full-time employment, part-time employment of 5–36 hours is associated with slightly lower satisfaction (−0.065 points, p = 0.002), while part-time employment of 1–4 hours is associated with higher satisfaction (+0.240 points, p = 0.001).

The model explains approximately 3.9% of the variation in job satisfaction (R² = 0.039).

These findings indicate statistically significant associations, not causal effects. The regression is unweighted and restricted to observations with valid employment-arrangement categories. Repeated observations are accounted for through respondent-clustered standard errors.



#### 6.8.3 RQ1.2 — Job Satisfaction Around Economic Events

This section tests whether average job satisfaction changed significantly around three major economic events:

- 2009: Global financial crisis
- 2015: Swiss franc shock
- 2020: COVID-19 pandemic

For each event, we compare the year before, the event year and the year after. We use OLS regression with respondent-clustered standard errors.

The analysis is descriptive and does not establish that the events caused changes in job satisfaction.


In [ ]:
# RQ1.2: Statistical tests around major economic events

events = {
    "2009 Financial Crisis": 2009,
    "2015 Swiss Franc Shock": 2015,
    "2020 COVID-19": 2020
}

event_results = []

for event_name, event_year in events.items():

    event_data = analysis_sample[
        analysis_sample["YEAR"].isin([
            event_year - 1,
            event_year,
            event_year + 1
        ])
    ][["IDPERS", "YEAR", "JOB_SATISFACTION"]].dropna().copy()

    # Reference category = year before the event
    event_data["PERIOD"] = pd.Categorical(
        event_data["YEAR"].map({
            event_year - 1: "Before",
            event_year: "During",
            event_year + 1: "After"
        }),
        categories=["Before", "During", "After"]
    )

    model = smf.ols(
        "JOB_SATISFACTION ~ C(PERIOD, Treatment(reference='Before'))",
        data=event_data
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": event_data["IDPERS"]}
    )

    print(f"\n{'=' * 55}")
    print(event_name)
    print(f"{'=' * 55}")
    print(model.summary())

    for term in model.params.index:
        if term == "Intercept":
            continue

        event_results.append({
            "Event": event_name,
            "Comparison": term,
            "Coefficient": model.params[term],
            "P_value": model.pvalues[term],
            "Observations": len(event_data)
        })

event_results_df = pd.DataFrame(event_results)

print("\nSummary of event comparisons:")
display(event_results_df.round(4))


#### Interpretation — Economic Events

We tested changes in job satisfaction around three major economic events using separate OLS regressions with respondent-clustered standard errors. Each regression compares the event year and following year with the preceding year.

**2009 Financial Crisis (2008–2010):**
Job satisfaction decreased by 0.034 points in 2009 compared with 2008, but the difference was not statistically significant (p = 0.147). In 2010, satisfaction was 0.053 points lower than in 2008 (p = 0.026).

**2015 Swiss Franc Shock (2014–2016):**
Job satisfaction was significantly lower in 2015 (−0.068 points, p < 0.001) and 2016 (−0.089 points, p < 0.001) compared with 2014.

**2020 COVID-19 (2019–2021):**
Job satisfaction increased significantly in 2020 compared with 2019 (+0.078 points, p < 0.001). In 2021, satisfaction was not significantly different from 2019 (+0.001 points, p = 0.977).

Overall, the three events show different patterns. The 2015 period shows consecutive declines, while the 2020 period shows a temporary increase.

Although some differences are statistically significant, their magnitudes are small. The regressions do not establish that the events caused these changes. They do not control for broader economic trends or changing respondent characteristics, and multiple comparisons were performed.



#### 6.8.4 RQ1.3 — Occupation × Economic Period Interaction

This section tests whether occupational differences in job satisfaction vary across four economic periods: 2004–2008, 2009–2014, 2015–2019 and 2020–2024.

We estimate an OLS regression with an interaction between occupation and economic period. Standard errors are clustered by respondent to account for repeated observations.

A joint Wald test evaluates whether the occupation × period interaction terms are statistically significant.

The results indicate associations rather than causal effects.


In [ ]:
# RQ1.3: Occupation × economic period interaction

rq13_data = analysis_sample[
    ["IDPERS", "YEAR", "OCCUPATION", "JOB_SATISFACTION"]
].dropna().copy()

# Keep civilian occupations only
rq13_data = rq13_data[
    rq13_data["OCCUPATION"].isin(range(1, 10))
].copy()

rq13_data["OCCUPATION"] = (
    rq13_data["OCCUPATION"].astype(int)
)

# Define economic periods
rq13_data["ECONOMIC_PERIOD"] = pd.cut(
    rq13_data["YEAR"],
    bins=[2003, 2008, 2014, 2019, 2024],
    labels=[
        "2004–2008",
        "2009–2014",
        "2015–2019",
        "2020–2024"
    ]
)

# OLS regression with interaction
model_rq13 = smf.ols(
    formula=(
        "JOB_SATISFACTION ~ "
        "C(OCCUPATION) * C(ECONOMIC_PERIOD)"
    ),
    data=rq13_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": rq13_data["IDPERS"]}
)

print(model_rq13.summary())

# Joint test of interaction terms
interaction_terms = [
    name for name in model_rq13.params.index
    if ":" in name
]

hypothesis = " = 0, ".join(interaction_terms) + " = 0"

interaction_test = model_rq13.wald_test(
    hypothesis,
    scalar=True
)

print("\nJoint occupation × period interaction test:")
print(interaction_test)

print("\nObservations:", len(rq13_data))
print("Unique respondents:", rq13_data["IDPERS"].nunique())


#### Interpretation — Occupation × Economic Period Interaction

The regression includes 124,635 observations from 22,242 respondents across nine civilian occupational groups and four economic periods.

The joint Wald test indicates that occupational differences in job satisfaction vary significantly across economic periods (χ²(24) = 45.88, p = 0.0046).

Using managers and 2004–2008 as reference categories, the clearest individual interaction differences appear in 2020–2024:

- Professionals: −0.190 points (p = 0.004).
- Technicians: −0.161 points (p = 0.022).
- Service and sales workers: −0.174 points (p = 0.023).

These coefficients indicate that job satisfaction in these occupations declined relative to managers when comparing 2020–2024 with 2004–2008.

The model explains approximately 0.9% of the variation in job satisfaction (R² = 0.009).

Overall, the findings suggest that occupational differences are not entirely stable over time. However, the estimated differences are relatively small, and the results do not establish that economic events caused these changes.

The analysis is unweighted, and standard errors are clustered by respondent. Individual interaction coefficients should be interpreted cautiously because multiple comparisons were performed.



## Summary — Key Findings and Next Steps

### 1. Dataset Preparation

We downloaded and processed Swiss Household Panel (SHP) data covering 2004–2024.

The individual and household datasets were merged, and the main variables were standardised across survey years.

**Final datasets:**

| Dataset | Observations | Unique respondents |
|---|---:|---:|
| Full dataset | 305,462 | 45,632 |
| Employed sample | 124,769 | 22,265 |
| Analysis sample | 124,766 | — |

The full dataset contains 10 standardised variables: respondent ID, household ID, year, age, sex, employment status, working status, occupation, job satisfaction and region of residence.

The employed sample includes respondents classified as employed (`WORKING_STATUS == 1`) with valid occupation and job satisfaction data.

The analysis sample additionally excludes observations with missing age or sex information.

No duplicate respondent-year combinations were identified in the full dataset.

### 2. Descriptive Findings

#### Overall Job Satisfaction

Average job satisfaction declined from approximately 8.21 in 2004 to 7.82 in 2024, a decrease of around 0.39 points on the 0–10 scale.

The median remained at 8, while the proportion of respondents reporting low job satisfaction (0–5) increased from 6.27% to 7.56%.

#### RQ1.1 — Occupational, Demographic and Employment Differences

**Occupation:**

Agricultural workers reported the highest average job satisfaction (8.34), while clerical support workers reported the lowest average among civilian occupations (7.78).

All nine civilian occupational groups experienced declines between 2004 and 2024, ranging from approximately 0.16 to 0.53 points.

**Age:**

Respondents aged 25–34 reported the lowest average satisfaction (7.66), while employed respondents aged 65+ reported the highest (8.78).

**Sex:**

Women reported slightly higher average job satisfaction than men across the full period (8.00 compared with 7.94).

**Region of residence:**

Eastern Switzerland had the highest average job satisfaction (8.13), while the Lake Geneva region had the lowest (7.80).

**Employment arrangements:**

A separate comparison included 104,638 observations with valid employment-arrangement categories.

| Employment arrangement | Observations | Mean satisfaction |
|---|---:|---:|
| Full-time (37+ hours) | 59,727 | 7.91 |
| Part-time (5–36 hours) | 43,189 | 7.96 |
| Part-time (1–4 hours) | 1,025 | 8.37 |
| Family business | 697 | 8.47 |

These are unadjusted averages and do not account for differences in respondent characteristics.

#### RQ1.2 — Economic Events

The descriptive analysis examined changes in job satisfaction around three major events:

- **2009 Financial Crisis:** Satisfaction decreased slightly between 2008 and 2010.
- **2015 Swiss Franc Shock:** Satisfaction declined in 2015 and remained lower in 2016.
- **2020 COVID-19:** Satisfaction increased temporarily in 2020 before returning close to its 2019 level in 2021.

These comparisons describe changes over time and do not establish that the events caused them.

#### RQ1.3 — Occupation × Economic Period

The descriptive analysis compared occupational differences across four periods:

- 2004–2008
- 2009–2014
- 2015–2019
- 2020–2024

The results showed that job satisfaction declined across occupational groups, but the size of the decline differed between occupations.

Professionals, technicians and service and sales workers showed relatively pronounced changes across periods.

### 3. Survey Weight Verification

Individual cross-sectional survey weights were identified and verified for all 21 years.

Of the 124,769 employed observations, 121,547 (97.42%) had positive weights.

**Weighted vs. unweighted comparison:**

| Measure | 2004 | 2024 | Change |
|---|---:|---:|---:|
| Unweighted mean | 8.212 | 7.815 | −0.397 |
| Weighted mean | 8.207 | 7.711 | −0.496 |

Both methods confirm the downward trend in job satisfaction.

The weighted comparison is included as a robustness check. The original descriptive analysis and regression models remain unweighted.

### 4. Statistical Testing

We conducted OLS regressions with standard errors clustered by respondent ID to account for repeated observations of the same individuals.

These tests examine statistical associations rather than causal relationships.

#### RQ1.1 — Occupational, Demographic and Employment Differences

An initial regression compared nine civilian occupational groups while controlling for survey year.

The joint test confirmed statistically significant occupational differences:

- Observations: 124,635
- Joint Wald test: χ²(8) = 168.22
- p < 0.001
- R² = 0.010

Using clerical support workers as the reference group, agricultural workers (+0.560), managers (+0.317) and professionals (+0.231) reported significantly higher job satisfaction.

A second regression included occupation, age, sex, region of residence, employment arrangement and survey year.

**Model results:**

- Observations: 104,526
- Unique respondents: 18,363
- R² = 0.039

**Joint Wald tests:**

| Variable | Chi-square | p-value |
|---|---:|---:|
| Occupation | 116.02 | <0.001 |
| Age group | 670.26 | <0.001 |
| Sex | 46.45 | <0.001 |
| Region of residence | 119.89 | <0.001 |
| Employment arrangement | 27.49 | <0.001 |

All five variables were jointly statistically significant after controlling for the other included characteristics and survey year.

Notable adjusted differences included:

- Clerical support workers reported lower satisfaction than managers (−0.321 points).
- Respondents aged 65+ reported higher satisfaction than those under 25 (+1.088 points).
- Sex category 2 reported higher satisfaction than category 1 (+0.156 points).
- Part-time employment of 5–36 hours was associated with slightly lower satisfaction than full-time employment (−0.065 points).

The regression sample was smaller because it required valid employment-arrangement categories.

#### RQ1.2 — Economic Events

Separate regressions compared job satisfaction before, during and after each economic event.

| Event | During vs. before | p-value | After vs. before | p-value |
|---|---:|---:|---:|---:|
| 2009 Financial Crisis | −0.034 | 0.147 | −0.053 | 0.026 |
| 2015 Swiss Franc Shock | −0.068 | <0.001 | −0.089 | <0.001 |
| 2020 COVID-19 | +0.078 | <0.001 | +0.001 | 0.977 |

The 2009 event-year difference was not statistically significant, although satisfaction was slightly lower in 2010 than in 2008.

The 2015 period showed statistically significant declines in both 2015 and 2016 compared with 2014.

The 2020 period showed a statistically significant temporary increase, followed by a return close to the 2019 level in 2021.

Although some differences were statistically significant, their magnitudes were small. The results do not establish that the economic events caused the changes.

#### RQ1.3 — Occupation × Economic Period Interaction

We estimated an OLS regression with an interaction between occupation and economic period.

**Model results:**

- Observations: 124,635
- Unique respondents: 22,242
- R² = 0.009
- Joint Wald test: χ²(24) = 45.88
- p = 0.0046

The joint interaction test was statistically significant, indicating that occupational differences in job satisfaction varied across economic periods.

Using managers and 2004–2008 as reference categories, the clearest individual interaction differences appeared in 2020–2024:

- Professionals: −0.190 points (p = 0.004).
- Technicians: −0.161 points (p = 0.022).
- Service and sales workers: −0.174 points (p = 0.023).

These results suggest that occupational differences were not completely stable over time. However, the individual interactions should be interpreted cautiously because multiple comparisons were performed.

### 5. Research Question Coverage

| Research question | Descriptive analysis | Statistical testing | Status |
|---|---|---|---|
| RQ1.1 — Differences by occupation, age, sex, region and employment arrangement | Completed | Completed | Initial analysis complete |
| RQ1.2 — Changes around major economic events | Completed | Completed | Initial analysis complete |
| RQ1.3 — Occupational differences across economic periods | Completed | Completed | Initial analysis complete |

All three research questions have now been addressed through descriptive analysis and preliminary statistical testing.

### 6. Methodological Limitations

**Survey weights:**
The main descriptive analysis and regressions are unweighted. A separate weighted comparison was conducted using individual cross-sectional weights, but subgroup-specific adjustments and the full complex survey design were not incorporated.

**Repeated respondents:**
The same individuals may appear in multiple survey years. Regression standard errors were clustered by respondent, but the models do not fully account for all aspects of the longitudinal survey design.

**Sample composition:**
The employed sample may change over time because of respondent attrition, survey refreshment and changes in employment status. These changes may influence observed trends.

**Missing and restricted observations:**
Some analyses use smaller samples because they require additional valid variables. In particular, the employment-arrangement regression uses 104,526 observations rather than the full employed sample of 124,769.

**Statistical significance:**
Large sample sizes can produce statistically significant results even when estimated differences are small. Model R² values are also relatively low, indicating limited explanatory power.

**Multiple comparisons:**
Several coefficients and event comparisons were tested. Individual p-values should therefore be interpreted cautiously.

**Causality:**
The analyses identify associations and changes over time. They do not establish that occupation, demographic characteristics or economic events caused differences in job satisfaction.

### 7. Next Steps

1. Review the notebook and ensure all descriptive and statistical results are correctly documented.
2. Confirm variable labels and coding against the SHP documentation.
3. Prepare the Quarto progress report using the validated findings.
4. Select the most relevant tables and figures for the report rather than including every notebook output.
5. Review the report with group members and incorporate feedback.
6. Consider additional robustness checks or methodological improvements before the final project submission.


## 7. Exporting the Processed Datasets

### Purpose

Save the cleaned datasets as CSV files so they can be reused for further analysis without repeating the data preparation process.

Three datasets are saved:

- `full_dataset.csv`: All respondents from 2004–2024.
- `employed_sample.csv`: Employed respondents with valid occupation and job satisfaction data.
- `analysis_sample.csv`: The employed sample after excluding observations with missing age or sex.

The files are saved in `data/processed`.

In [ ]:

from pathlib import Path

# Define the folder for processed datasets
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

# Save the full dataset containing all survey respondents
full_dataset.to_csv(
    output_folder / "full_dataset.csv",
    index=False
)

# Save employed respondents with valid research variables
employed_sample.to_csv(
    output_folder / "employed_sample.csv",
    index=False
)

# Save employed respondents with complete demographic information
analysis_sample.to_csv(
    output_folder / "analysis_sample.csv",
    index=False
)

# Confirm that all three datasets have been saved
print("Datasets saved successfully.")
print("Full dataset:", full_dataset.shape)
print("Employed sample:", employed_sample.shape)
print("Demographic analysis sample:", analysis_sample.shape)


In [ ]:
# Verify that the exported CSV files match the original datasets

datasets = {
    "full_dataset.csv": full_dataset,
    "employed_sample.csv": employed_sample,
    "analysis_sample.csv": analysis_sample
}

for filename, original in datasets.items():
    file_path = output_folder / filename

    assert file_path.is_file(), f"Missing file: {filename}"

    exported = pd.read_csv(file_path)

    assert exported.shape == original.shape, (
        f"Shape mismatch in {filename}"
    )

    assert exported.columns.tolist() == original.columns.tolist(), (
        f"Column mismatch in {filename}"
    )

    print(f"{filename}: verified {exported.shape}")